In [1]:
import pandas as pd
import numpy as np 
from scipy import stats as stat
from scipy.stats import bootstrap as bootstrap
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import catplot
import random as rd
import os
from sklearn.preprocessing import StandardScaler as ss
from sklearn.decomposition import PCA as pca
import math
from scipy.stats import linregress
import mne
%matplotlib qt
import os
from itertools import compress

import mne
from mne import datasets
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
import mne_bids
# Import StatsModels
import statsmodels.formula.api as smf
import matplotlib as mpl

# Import MNE processing
from mne.preprocessing.nirs import beer_lambert_law, optical_density

# Import MNE-BIDS processing
from mne_bids import BIDSPath, get_entity_vals, read_raw_bids,stats

from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm, statsmodels_to_results
from mne_nirs.visualisation import plot_glm_group_topo, plot_glm_surface_projection 
import os.path as op

from mne.datasets.testing import data_path
from mne.viz import set_3d_view

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm
# Import common libraries
from collections import defaultdict
from copy import deepcopy
from itertools import compress
from pprint import pprint

# Import Plotting Library
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import StatsModels
import statsmodels.formula.api as smf
from mne import Epochs, events_from_annotations, set_log_level
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    temporal_derivative_distribution_repair,
)

# Import MNE processing
from mne.viz import plot_compare_evokeds

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids

# Import MNE-NIRS processing
from mne_nirs.channels import get_long_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.signal_enhancement import enhance_negative_correlation


C:\Users\pkell\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [550]:
## Split Half Analysis by Condition - Group Level 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\1st-2nd Half Split\Channel\1st_half_GLM_GROUP.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\1st-2nd Half Split\Channel\2nd_half_GLM_GROUP.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Combine Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Initialize Results ---
condition_results = []

# --- Run Split-Half Analysis Per Condition ---
for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]

    session1 = cond_data[cond_data['session'] == 1]
    session2 = cond_data[cond_data['session'] == 2]

    # Match on channel
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for condition {condition}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        sb_corrected = (2 * r) / (1 + r) if r < 1 else 1.0  # cap at 1
        condition_results.append({
            'condition': condition,
            'pearson_r': r,
            'spearman_brown': sb_corrected,
            'n_channels': merged.shape[0],
            'p_value': p
        })
    except Exception as e:
        print(f"❌ Error computing r for condition {condition}: {e}")

# --- Results DataFrame ---
group_corr_df = pd.DataFrame(condition_results)
print(group_corr_df)

# --- Plot ---
plt.figure(figsize=(6, 4))
sns.barplot(data=group_corr_df, x='condition', y='spearman_brown', palette='muted', edgecolor='black')

# Add text annotations
for i, row in group_corr_df.iterrows():
    plt.text(i, row['spearman_brown'] + 0.02, f"{row['spearman_brown']:.2f}", ha='center', va='bottom', fontsize=10)

plt.title('Split-Half Reliability (1st vs 2nd Half) by Condition')
plt.xlabel('Condition')
plt.ylabel('Spearman-Brown Corrected r')
plt.ylim(0, max(group_corr_df['spearman_brown']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()
# --- Plot Uncorrected Pearson r ---
plt.figure(figsize=(6, 4))
sns.barplot(data=group_corr_df, x='condition', y='pearson_r', palette='muted', edgecolor='black')

# Add text annotations
for i, row in group_corr_df.iterrows():
    plt.text(i, row['pearson_r'] + 0.02, f"{row['pearson_r']:.2f}", ha='center', va='bottom', fontsize=10)

plt.title('Split-Half Correlation (1st vs 2nd Half) by Condition')
plt.xlabel('Condition')
plt.ylabel('Pearson r')
plt.ylim(0, max(group_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()



  condition  pearson_r  spearman_brown  n_channels   p_value
0        x1   0.087864        0.161536          46  0.561476
1        x2   0.084311        0.155511          46  0.577472


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\527046880.py:67: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=group_corr_df, x='condition', y='spearman_brown', palette='muted', edgecolor='black')
C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\527046880.py:82: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=group_corr_df, x='condition', y='pearson_r', palette='muted', edgecolor='black')


In [568]:
import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\Interleaved Block Split\Channel\even_contrast.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\Interleaved Block Split\Channel\odd_contrast.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Combine Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Initialize Results ---
condition_results = []

# --- Run Split-Half Analysis Per Condition ---
for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]

    session1 = cond_data[cond_data['session'] == 1]
    session2 = cond_data[cond_data['session'] == 2]

    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for condition {condition}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        sb_corrected = (2 * r) / (1 + r) if r < 1 else 1.0
        var1 = merged['tstat_s1'].var()
        var2 = merged['tstat_s2'].var()
        var_stability = min(var1, var2) / max(var1, var2) if max(var1, var2) > 0 else 0

        condition_results.append({
            'condition': condition,
            'pearson_r': r,
            'spearman_brown': sb_corrected,
            'variance_stability': var_stability,
            'n_channels': merged.shape[0],
            'p_value': p
        })
    except Exception as e:
        print(f"❌ Error computing r for condition {condition}: {e}")

# --- Convert to DataFrame ---
group_corr_df = pd.DataFrame(condition_results)
print(group_corr_df)

# --- Reshape for Grouped Bar Plot ---
plot_df = pd.melt(
    group_corr_df,
    id_vars='condition',
    value_vars=['pearson_r', 'spearman_brown', 'variance_stability'],
    var_name='Metric',
    value_name='Value'
)

# --- Create Grouped Bar Plot ---
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='condition', y='Value', hue='Metric', palette='pastel', edgecolor='black')

# Add value annotations to each bar
for container in plt.gca().containers:
    for bar in container:
        height = bar.get_height()
        if not pd.isna(height):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.02,
                f'{height:.2f}',
                ha='center',
                va='bottom',
                fontsize=9
            )

plt.title('Split-Half Metrics by Condition')
plt.xlabel('Condition')
plt.ylabel('Metric Value')
plt.ylim(0, 1.2)
plt.legend(title='Metric')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()



  condition  pearson_r  spearman_brown  variance_stability  n_channels  \
0    -x1+x2  -0.027248       -0.056023            0.827301          46   

    p_value  
0  0.857346  


In [570]:
## Group Level sp;it half correlation - filter by significant channels 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\Interleaved Block Split\Channel\even_contrast.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\Interleaved Block Split\Channel\odd_contrast.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Filter by q-value threshold ---
df1 = df1[df1['q'] <= 0.06]
df2 = df2[df2['q'] <= 0.06]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Combine Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Initialize Results ---
condition_results = []

# --- Run Split-Half Analysis Per Condition ---
for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]

    session1 = cond_data[cond_data['session'] == 1]
    session2 = cond_data[cond_data['session'] == 2]

    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for condition {condition}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        sb_corrected = (2 * r) / (1 + r) if r < 1 else 1.0
        var1 = merged['tstat_s1'].var()
        var2 = merged['tstat_s2'].var()
        var_stability = min(var1, var2) / max(var1, var2) if max(var1, var2) > 0 else 0

        condition_results.append({
            'condition': condition,
            'pearson_r': r,
            'spearman_brown': sb_corrected,
            'variance_stability': var_stability,
            'n_channels': merged.shape[0],
            'p_value': p
        })
    except Exception as e:
        print(f"❌ Error computing r for condition {condition}: {e}")

# --- Convert to DataFrame ---
group_corr_df = pd.DataFrame(condition_results)
print(group_corr_df)

# --- Reshape for Grouped Bar Plot ---
plot_df = pd.melt(
    group_corr_df,
    id_vars='condition',
    value_vars=['pearson_r', 'spearman_brown', 'variance_stability'],
    var_name='Metric',
    value_name='Value'
)

# --- Create Grouped Bar Plot ---
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='condition', y='Value', hue='Metric', palette='pastel', edgecolor='black')

# Add value annotations to each bar
for container in plt.gca().containers:
    for bar in container:
        height = bar.get_height()
        if not pd.isna(height):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.02,
                f'{height:.2f}',
                ha='center',
                va='bottom',
                fontsize=9
            )

plt.title('Split-Half Metrics by Condition')
plt.xlabel('Condition')
plt.ylabel('Metric Value')
plt.ylim(0, 1.2)
plt.legend(title='Metric')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


  condition  pearson_r  spearman_brown  variance_stability  n_channels  \
0    -x1+x2   0.044547        0.085294             0.62899          12   

    p_value  
0  0.890662  


In [504]:
## Split Half Analysis by Condition - Subject Level - Condition

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Subject-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\1st_glm_result_subjectlevel.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\2nd_glm_result_subjectlevel_ROI_AGG.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Label halves ---
df1['half'] = 'even'
df2['half'] = 'odd'

# --- Combine ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Per Subject/Condition Correlation ---
subject_results = []

for (subj, cond), group in combined.groupby(['subject', 'cond']):
    even = group[group['half'] == 'even']
    odd = group[group['half'] == 'odd']

    merged = pd.merge(
        even, odd,
        on=['source', 'detector'],
        suffixes=('_even', '_odd')
    )

    if merged.shape[0] < 3:
        continue

    try:
        r, _ = pearsonr(merged['tstat_even'], merged['tstat_odd'])
        sb_corr = (2 * r) / (1 + r) if r < 1 else 1.0
        subject_results.append({
            'subject': subj,
            'condition': cond,
            'pearson_r': r,
            'spearman_brown': sb_corr
        })
    except:
        continue

# --- Convert to DataFrame ---
subject_df = pd.DataFrame(subject_results)

# --- Summary Stats ---
summary_df = subject_df.groupby('condition').agg(
    mean_pearson_r=('pearson_r', 'mean'),
    mean_spearman_brown=('spearman_brown', 'mean'),
    n=('subject', 'count')
).reset_index()

print(subject_df)

# --- Plot Subject-Level r and SB-corrected r ---
plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='pearson_r', hue='subject', dodge=True)
plt.title('Subject-Level Split-Half Pearson r (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Pearson r')
plt.ylim(0, 1.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='spearman_brown', hue='subject', dodge=True)
plt.title('Subject-Level Spearman-Brown Corrected r (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Spearman-Brown Corrected r')
plt.ylim(0, 1.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

# --- Reshape to Long Format for Plotting Mean r and SB ---
plot_df = pd.melt(
    subject_df,
    id_vars=['condition'],
    value_vars=['pearson_r', 'spearman_brown'],
    var_name='Metric',
    value_name='r_value'
)

# --- Compute Mean per Condition and Metric ---
mean_df = plot_df.groupby(['condition', 'Metric'], as_index=False).mean()

# --- Plot Mean Pearson r and SB-Corrected r Side-by-Side ---
plt.figure(figsize=(8, 5))
sns.barplot(data=mean_df, x='condition', y='r_value', hue='Metric', palette='pastel', edgecolor='black')

# Annotate bars with values
for i, row in mean_df.iterrows():
    plt.text(
        x=i % len(mean_df['condition'].unique()) + (0.15 if row['Metric'] == 'spearman_brown' else -0.15),
        y=row['r_value'] + 0.015,
        s=f"{row['r_value']:.2f}",
        ha='center',
        va='bottom',
        fontsize=10
    )

plt.title('Mean Split-Half Pearson r and Spearman-Brown Corrected r')
plt.ylabel('Mean r Value')
plt.xlabel('Condition')
plt.ylim(0, 1.5)
plt.legend(title='Metric')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


                         subject condition  pearson_r  spearman_brown
0                   allison2 rel        x1   0.075691        0.140731
1                   allison2 rel        x2   0.523487        0.687222
2                      pid001rel        x1   0.221664        0.362889
3                      pid001rel        x2   0.172535        0.294294
4                      pid002rel        x1   0.138115        0.242708
5                      pid002rel        x2   0.736462        0.848233
6                      pid003rel        x1   0.157949        0.272808
7                      pid003rel        x2  -0.121227       -0.275900
8                      pid004rel        x1   0.290426        0.450125
9                      pid004rel        x2   0.098508        0.179348
10                     pid005rel        x1   0.276197        0.432843
11                     pid005rel        x2  -0.147333       -0.345582
12                     pid006rel        x1   0.363463        0.533147
13                  

C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1715950180.py:74: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1715950180.py:84: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()


In [506]:
## Split Half Analysis by Condition - Subject Level Task 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Subject-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\1st_glm_result_subjectlevel.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\2nd_glm_result_subjectlevel_ROI_AGG.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Label halves ---
df1['half'] = 'even'
df2['half'] = 'odd'

# --- Combine ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Per Subject/Condition Correlation ---
subject_results = []

for (subj, cond), group in combined.groupby(['subject', 'cond']):
    even = group[group['half'] == 'even']
    odd = group[group['half'] == 'odd']

    merged = pd.merge(
        even, odd,
        on=['source', 'detector'],
        suffixes=('_even', '_odd')
    )

    if merged.shape[0] < 3:
        continue

    try:
        r, _ = pearsonr(merged['tstat_even'], merged['tstat_odd'])
        sb_corr = (2 * r) / (1 + r) if r < 1 else 1.0
        subject_results.append({
            'subject': subj,
            'condition': cond,
            'pearson_r': r,
            'spearman_brown': sb_corr
        })
    except:
        continue

# --- Convert to DataFrame ---
subject_df = pd.DataFrame(subject_results)

# --- Summary Stats ---
summary_df = subject_df.groupby('condition').agg(
    mean_pearson_r=('pearson_r', 'mean'),
    mean_spearman_brown=('spearman_brown', 'mean'),
    n=('subject', 'count')
).reset_index()

print(subject_df)
print(summary_df)

# --- Plot Subject-Level r and SB-corrected r ---
plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='pearson_r', hue='subject', dodge=True)
plt.title('Subject-Level Split-Half Pearson r (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Pearson r')
plt.ylim(0, 1.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='spearman_brown', hue='subject', dodge=True)
plt.title('Subject-Level Spearman-Brown Corrected r (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Spearman-Brown Corrected r')
plt.ylim(0, 1.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

# --- Reshape to Long Format for Plotting Mean r and SB ---
plot_df = pd.melt(
    subject_df,
    id_vars=['condition'],
    value_vars=['pearson_r', 'spearman_brown'],
    var_name='Metric',
    value_name='r_value'
)

# --- Compute Mean per Condition and Metric ---
mean_df = plot_df.groupby(['condition', 'Metric'], as_index=False).mean()

# --- Plot Mean Pearson r and SB-Corrected r Side-by-Side ---
plt.figure(figsize=(8, 5))
sns.barplot(data=mean_df, x='condition', y='r_value', hue='Metric', palette='pastel', edgecolor='black')

# Annotate bars with values
for i, row in mean_df.iterrows():
    plt.text(
        x=i % len(mean_df['condition'].unique()) + (0.15 if row['Metric'] == 'spearman_brown' else -0.15),
        y=row['r_value'] + 0.015,
        s=f"{row['r_value']:.2f}",
        ha='center',
        va='bottom',
        fontsize=10
    )

plt.title('Mean Split-Half Pearson r and Spearman-Brown Corrected r')
plt.ylabel('Mean r Value')
plt.xlabel('Condition')
plt.ylim(0, 1.5)
plt.legend(title='Metric')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

                         subject condition  pearson_r  spearman_brown
0                   allison2 rel        x1   0.075691        0.140731
1                   allison2 rel        x2   0.523487        0.687222
2                      pid001rel        x1   0.221664        0.362889
3                      pid001rel        x2   0.172535        0.294294
4                      pid002rel        x1   0.138115        0.242708
5                      pid002rel        x2   0.736462        0.848233
6                      pid003rel        x1   0.157949        0.272808
7                      pid003rel        x2  -0.121227       -0.275900
8                      pid004rel        x1   0.290426        0.450125
9                      pid004rel        x2   0.098508        0.179348
10                     pid005rel        x1   0.276197        0.432843
11                     pid005rel        x2  -0.147333       -0.345582
12                     pid006rel        x1   0.363463        0.533147
13                  

C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1960481661.py:75: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1960481661.py:85: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()


In [574]:
## Split Half Variance

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# # --- Load Subject-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_subject_level_contrasts.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_subject_level_contrasts.csv"
# --- Load Subject-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_subject_level_contrasts_ROI_AGG.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_subject_level_contrasts_ROI_AGG.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Label halves ---
df1['half'] = 'even'
df2['half'] = 'odd'

# --- Combine ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Per Subject/Condition Analysis ---
subject_results = []

for (subj, cond), group in combined.groupby(['subject', 'cond']):
    even = group[group['half'] == 'even']
    odd = group[group['half'] == 'odd']

    merged = pd.merge(
        even, odd,
        on=['source', 'detector'],
        suffixes=('_even', '_odd')
    )

    if merged.shape[0] < 3:
        continue

    try:
        # Correlation
        r, _ = pearsonr(merged['tstat_even'], merged['tstat_odd'])
        sb_corr = (2 * r) / (1 + r) if r < 1 else 1.0

        # Variance Stability = min(var1, var2) / max(var1, var2)
        var_even = merged['tstat_even'].var()
        var_odd = merged['tstat_odd'].var()
        if var_even > 0 and var_odd > 0:
            var_stability = min(var_even, var_odd) / max(var_even, var_odd)
        else:
            var_stability = 0  # degenerate case

        subject_results.append({
            'subject': subj,
            'condition': cond,
            'pearson_r': r,
            'spearman_brown': sb_corr,
            'variance_stability': var_stability
        })
    except:
        continue

# --- Convert to DataFrame ---
subject_df = pd.DataFrame(subject_results)

# --- Summary Stats ---
summary_df = subject_df.groupby('condition').agg(
    mean_pearson_r=('pearson_r', 'mean'),
    mean_spearman_brown=('spearman_brown', 'mean'),
    mean_variance_stability=('variance_stability', 'mean'),
    n=('subject', 'count')
).reset_index()

print(summary_df)
print(subject_df)


# --- Plot Pearson r ---
plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='pearson_r', hue='subject', dodge=True)
plt.title('Subject-Level Split-Half Pearson r (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Pearson r')
plt.ylim(0, 1.2)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

# --- Plot Spearman-Brown corrected r ---
plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='spearman_brown', hue='subject', dodge=True)
plt.title('Subject-Level Spearman-Brown Corrected r (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Spearman-Brown Corrected r')
plt.ylim(0, 1.2)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

# --- Plot Variance Stability ---
plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='variance_stability', hue='subject', dodge=True)
plt.title('Subject-Level Variance Stability (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Variance Stability (min/max)')
plt.ylim(0, 1.2)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

# --- Mean Plot: Pearson r, SB, and Variance Stability ---
plot_df = pd.melt(
    subject_df,
    id_vars=['condition'],
    value_vars=['pearson_r', 'spearman_brown', 'variance_stability'],
    var_name='Metric',
    value_name='r_value'
)

mean_df = plot_df.groupby(['condition', 'Metric'], as_index=False).mean()

plt.figure(figsize=(8, 5))
sns.barplot(data=mean_df, x='condition', y='r_value', hue='Metric', palette='pastel', edgecolor='black')

for i, row in mean_df.iterrows():
    plt.text(
        x=i % len(mean_df['condition'].unique()) + (0.15 if row['Metric'] != 'pearson_r' else -0.15),
        y=row['r_value'] + 0.02,
        s=f"{row['r_value']:.2f}",
        ha='center', va='bottom', fontsize=10
    )

plt.title('Mean Split-Half Metrics by Condition')
plt.ylabel('Mean Value')
plt.xlabel('Condition')
plt.ylim(0, 1.2)
plt.legend(title='Metric')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


  condition  mean_pearson_r  mean_spearman_brown  mean_variance_stability   n
0     x1-x2       -0.007987            -0.445311                 0.543047  73
                                              subject condition  pearson_r  \
0   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2  -0.097868   
1   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2   0.432044   
2   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2  -0.629652   
3   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2  -0.134236   
4   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2   0.259221   
5   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2  -0.324474   
6   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2   0.197293   
7   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2   0.242090   
8   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2  -0.700002   
9   C:\Users\pkell\OneDrive\Documents\fNIRS\pilot ...     x1-x2 

C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\2203546220.py:91: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\2203546220.py:102: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\2203546220.py:113: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()


In [28]:
## Split Half Task vs Rest Variance 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Subject-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_blocks_glm_result_subjectlevel_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_blocks_glm_result_subjectlevel_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Label halves ---
df1['half'] = 'even'
df2['half'] = 'odd'

# --- Combine ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Per Subject Correlation and Variance Stability ---
subject_results = []

for subj, group in combined.groupby('subject'):
    even = group[group['half'] == 'even']
    odd = group[group['half'] == 'odd']

    merged = pd.merge(
        even, odd,
        on=['source', 'detector'],
        suffixes=('_even', '_odd')
    )

    if merged.shape[0] < 3:
        continue

    try:
        # Correlation
        r, _ = pearsonr(merged['tstat_even'], merged['tstat_odd'])
        sb_corr = (2 * r) / (1 + r) if r < 1 else 1.0

        # Variance stability (min/max ratio)
        var_even = merged['tstat_even'].var()
        var_odd = merged['tstat_odd'].var()
        var_stability = min(var_even, var_odd) / max(var_even, var_odd) if max(var_even, var_odd) > 0 else 0

        subject_results.append({
            'subject': subj,
            'pearson_r': r,
            'spearman_brown': sb_corr,
            'variance_stability': var_stability
        })
    except:
        continue

# --- Convert to DataFrame ---
subject_df = pd.DataFrame(subject_results)

# --- Summary ---
mean_r = subject_df['pearson_r'].mean()
mean_sb = subject_df['spearman_brown'].mean()
mean_varstab = subject_df['variance_stability'].mean()

print(f"Mean Pearson r: {mean_r:.3f}")
print(f"Mean Spearman-Brown corrected r: {mean_sb:.3f}")
print(f"Mean Variance Stability: {mean_varstab:.3f}")

# --- Barplot: Subject-Level r, SB, Variance Stability ---
plot_df = pd.melt(subject_df, id_vars=['subject'], 
                  value_vars=['pearson_r', 'spearman_brown', 'variance_stability'],
                  var_name='Metric', value_name='r_value')

plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x='subject', y='r_value', hue='Metric', palette='pastel', edgecolor='black')
plt.title('Subject-Level Split-Half Reliability and Variance Stability (Odd vs Even Blocks)')
plt.xlabel('Subject')
plt.ylabel('Value')
plt.ylim(0, 1.2)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# --- Barplot: Group Means ---
mean_df = plot_df.groupby('Metric', as_index=False)['r_value'].mean()

plt.figure(figsize=(6, 5))
sns.barplot(data=mean_df, x='Metric', y='r_value', palette='muted', edgecolor='black')

# Annotate values
for i, row in mean_df.iterrows():
    plt.text(i, row['r_value'] + 0.02, f"{row['r_value']:.2f}", ha='center', va='bottom', fontsize=10)

plt.title('Mean Split-Half Metrics Across Subjects')
plt.ylabel('Mean Value')
plt.ylim(0, 1.2)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


Mean Pearson r: 1.000
Mean Spearman-Brown corrected r: 1.000
Mean Variance Stability: 1.000


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\244868340.py:92: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=mean_df, x='Metric', y='r_value', palette='muted', edgecolor='black')


In [208]:
## Condition 1/2 group level correlation code 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_glm_result_group_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_glm_result_group_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Combine Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Initialize Results ---
condition_results = []

# --- Run Analysis Per Condition ---
for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]

    session1 = cond_data[cond_data['session'] == 1]
    session2 = cond_data[cond_data['session'] == 2]

    # Match on channel
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],  # ensure channels match
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for condition {condition}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        condition_results.append({
            'condition': condition,
            'pearson_r': r,
            'n_channels': merged.shape[0],
            'p_value': p
        })
    except Exception as e:
        print(f"❌ Error computing r for condition {condition}: {e}")

# --- Results DataFrame ---
group_corr_df = pd.DataFrame(condition_results)
print(group_corr_df)

# --- Plot ---
plt.figure(figsize=(6, 4))
sns.barplot(data=group_corr_df, x='condition', y='pearson_r', palette='muted', edgecolor='black')

# Add text annotations
for i, row in group_corr_df.iterrows():
    plt.text(i, row['pearson_r'] + 0.02, f"{row['pearson_r']:.2f}", ha='center', va='bottom', fontsize=10)

plt.title('Group-Level Test–Retest Correlation by Condition')
plt.xlabel('Condition')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.ylim(0, max(group_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()



  condition  pearson_r  n_channels   p_value
0        x1   0.721753          10  0.018443
1        x2   0.733212          10  0.015827


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\4015273567.py:65: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=group_corr_df, x='condition', y='pearson_r', palette='muted', edgecolor='black')


In [541]:
## Group Level Task v Rest Correlation
import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_Group_glm_result_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_Group_glm_result_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Match channels by source and detector across sessions ---
merged = pd.merge(
    df1,
    df2,
    on=['source', 'detector'],
    suffixes=('_s1', '_s2')
)

if merged.shape[0] < 3:
    print("⚠️ Not enough matched channels between sessions. Exiting.")
else:
    # Compute Pearson correlation between t-values
    r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])

    print(f"✅ Pearson r (Session 1 vs. Session 2 t-values): {r:.2f}, p-value: {p:.2g}")
    print(f"Matched channels: {merged.shape[0]}")

    # --- Plotting ---
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=merged['tstat_s1'], y=merged['tstat_s2'], s=60, color='steelblue', edgecolor='black')
    
    # Line of best fit
    fit = np.polyfit(merged['tstat_s1'], merged['tstat_s2'], 1)
    fit_line = np.poly1d(fit)
    x_vals = np.linspace(merged['tstat_s1'].min(), merged['tstat_s1'].max(), 100)
    plt.plot(x_vals, fit_line(x_vals), linestyle='--', color='black')

    # Add correlation stats to plot
    plt.text(0.05, 0.95,
             f'r = {r:.2f}\np = {p:.2g}',
             transform=plt.gca().transAxes,
             verticalalignment='top',
             fontsize=11,
             bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title('Correlation of Channel Task vs Rest t-values: Session 1 vs Session 2', fontsize = 18)
    plt.xlabel('Session 1 t-values', fontsize = 16)
    plt.ylabel('Session 2 t-values', fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


✅ Pearson r (Session 1 vs. Session 2 t-values): 0.74, p-value: 3e-09
Matched channels: 46


In [537]:
## Group Level Contrast Correlation  
import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_Contrast_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_Contrast_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Match channels by source and detector across sessions ---
merged = pd.merge(
    df1,
    df2,
    on=['source', 'detector'],
    suffixes=('_s1', '_s2')
)

if merged.shape[0] < 3:
    print("⚠️ Not enough matched channels between sessions. Exiting.")
else:
    # Compute Pearson correlation between t-values
    r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])

    print(f"✅ Pearson r (Session 1 vs. Session 2 t-values): {r:.4f}, p-value: {p:.4g}")
    print(f"Matched channels: {merged.shape[0]}")

    # --- Plotting ---
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=merged['tstat_s1'], y=merged['tstat_s2'], s=60, color='steelblue', edgecolor='black')
    
    # Line of best fit
    fit = np.polyfit(merged['tstat_s1'], merged['tstat_s2'], 1)
    fit_line = np.poly1d(fit)
    x_vals = np.linspace(merged['tstat_s1'].min(), merged['tstat_s1'].max(), 100)
    plt.plot(x_vals, fit_line(x_vals), linestyle='--', color='black')

    # Add correlation stats to plot
    plt.text(0.05, 0.95,
             f'r = {r:.2f}\np = {p:.2g}',
             transform=plt.gca().transAxes,
             verticalalignment='top',
             fontsize=11,
             bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title('Correlation of Channel 2nd vs 1st t-values: Session 1 vs Session 2', fontsize = 18)
    plt.xlabel('Session 1 t-values', fontsize = 16)
    plt.ylabel('Session 2 t-values', fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

✅ Pearson r (Session 1 vs. Session 2 t-values): 0.0897, p-value: 0.5533
Matched channels: 46


In [545]:
## Group Level Contrast Correlation - Variance
import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_Contrast_glm_result_ROI_AGG_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_Contrast_glm_result_ROI_AGG_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Match channels by source and detector across sessions ---
merged = pd.merge(
    df1,
    df2,
    on=['source', 'detector'],
    suffixes=('_s1', '_s2')
)

if merged.shape[0] < 3:
    print("⚠️ Not enough matched channels between sessions. Exiting.")
else:
    # Compute squared t-values for variance estimate
    t2_s1 = merged['tstat_s1'] ** 2
    t2_s2 = merged['tstat_s2'] ** 2

    # Pearson correlation of variances
    r, p = pearsonr(t2_s1, t2_s2)

    print(f"✅ Pearson r (t² variance): {r:.2f}, p-value: {p:.2g}")
    print(f"Matched channels: {merged.shape[0]}")

    # --- Plotting ---
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=t2_s1, y=t2_s2, s=60, color='mediumseagreen', edgecolor='black')

    # Line of best fit
    fit = np.polyfit(merged['tstat_s1'], merged['tstat_s2'], 1)
    fit_line = np.poly1d(fit)
    x_vals = np.linspace(merged['tstat_s1'].min(), merged['tstat_s1'].max(), 100)
    plt.plot(x_vals, fit_line(x_vals), linestyle='--', color='black')


    # Add correlation stats to plot
    plt.text(0.05, 0.95,
             f'r = {r:.2f}\np = {p:.4g}',
             transform=plt.gca().transAxes,
             verticalalignment='top',
             fontsize=11,
             bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title('2nd vs 1st Contrast Variance Stability: Session 1 vs Session 2', fontsize=18)
    plt.xlabel('Session 1 Variance (t²)', fontsize=16)
    plt.ylabel('Session 2 Variance (t²)', fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

✅ Pearson r (t² variance): -0.02, p-value: 0.95
Matched channels: 10


In [212]:
## Condition 1/2 group level correlation code - VARIANCE 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_glm_result_group_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_glm_result_group_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Combine Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Initialize Results ---
condition_results = []

# --- Run Variance Stability Analysis Per Condition ---
for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]

    session1 = cond_data[cond_data['session'] == 1]
    session2 = cond_data[cond_data['session'] == 2]

    # Match channels
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],  # ensure channels match
        suffixes=('_s1', '_s2')
    ).dropna(subset=['tstat_s1', 'tstat_s2'])

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for condition {condition}. Skipping.")
        continue

    try:
        # Variance (squared t-values)
        var_s1 = merged['tstat_s1'] ** 2
        var_s2 = merged['tstat_s2'] ** 2

        # Correlation of variance vectors
        r, p = pearsonr(var_s1, var_s2)

        condition_results.append({
            'condition': condition,
            'pearson_r': r,
            'n_channels': merged.shape[0],
            'p_value': p
        })

    except Exception as e:
        print(f"❌ Error computing variance r for condition {condition}: {e}")

# --- Results DataFrame ---
group_var_corr_df = pd.DataFrame(condition_results)
print(group_var_corr_df)

# --- Plot ---
plt.figure(figsize=(6, 4))
sns.barplot(data=group_var_corr_df, x='condition', y='pearson_r', palette='muted', edgecolor='black')

# Add text annotations
for i, row in group_var_corr_df.iterrows():
    plt.text(i, row['pearson_r'] + 0.02, f"{row['pearson_r']:.2f}", ha='center', va='bottom', fontsize=10)

plt.title('Group-Level Variance Stability (R²) by Condition')
plt.xlabel('Condition')
plt.ylabel('Pearson r of Variance (Session 1 vs Session 2)')
plt.ylim(0, max(group_var_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


  condition  pearson_r  n_channels   p_value
0        x1   0.762546          10  0.010324
1        x2   0.738278          10  0.014757


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1717626744.py:72: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=group_var_corr_df, x='condition', y='pearson_r', palette='muted', edgecolor='black')


In [214]:
## Group Level Task v Rest Correlation - Variance 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_glm_result_group_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_glm_result_group_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Match channels by source and detector across sessions ---
merged = pd.merge(
    df1,
    df2,
    on=['source', 'detector'],
    suffixes=('_s1', '_s2')
)

if merged.shape[0] < 3:
    print("⚠️ Not enough matched channels between sessions. Exiting.")
else:
    # Compute squared t-values (proxy for variance)
    var_s1 = merged['tstat_s1'] ** 2
    var_s2 = merged['tstat_s2'] ** 2

    # Correlation between variances
    r, p = pearsonr(var_s1, var_s2)

    print(f"✅ Pearson r (Session 1 vs. Session 2 variance): {r:.4f}, p-value: {p:.4g}")
    print(f"Matched channels: {merged.shape[0]}")

    # --- Plotting ---
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=var_s1, y=var_s2, s=60, color='mediumseagreen', edgecolor='black')

    # Identity line
    plt.plot([var_s1.min(), var_s1.max()],
             [var_s1.min(), var_s1.max()],
             linestyle='--', color='gray')

    # Add r and p-value annotation
    plt.text(0.05, 0.95,
             f'r = {r:.2f}\np = {p:.4g}',
             transform=plt.gca().transAxes,
             verticalalignment='top',
             fontsize=11,
             bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title('Task vs Rest Variance Stability: Session 1 vs Session 2')
    plt.xlabel('Session 1 Variance (t²)')
    plt.ylabel('Session 2 Variance (t²)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()



✅ Pearson r (Session 1 vs. Session 2 variance): 0.8746, p-value: 0.0009281
Matched channels: 10


In [426]:
## Group Level Task vs Rest Correlation - Functional Connectivity - within session 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\73_even_group_functional_conn_ROI_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\73_odd_group_functional_conn_ROI_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Labels ---
df1['session'] = 1
df2['session'] = 2

# --- Merge on Channel Pair Info ---
merged = pd.merge(
    df1, df2,
    on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
    suffixes=('_s1', '_s2')
).dropna(subset=['R_s1', 'R_s2'])

# --- Compute Group-Level Correlation of R Values ---
r_val, p_val = pearsonr(merged['R_s1'], merged['R_s2'])

print(f"✅ Group-level R correlation across sessions: r = {r_val:.4f}, p = {p_val}")
print(f"Number of matched channel pairs: {len(merged)}")

# --- Plotting ---
plt.figure(figsize=(6, 4))
sns.scatterplot(x=merged['R_s1'], y=merged['R_s2'], s=60, color='mediumseagreen', edgecolor='black')
plt.plot(
    [merged['R_s1'].min(), merged['R_s1'].max()],
    [merged['R_s1'].min(), merged['R_s1'].max()],
    linestyle='--', color='red'
)
# title = "Session-to-Session Functional Connectivity (Channel) \"
plt.title(f"Within Session Functional Connectivity (Channel)\nr = {r_val:.2f}, p < .001", fontsize=12)
plt.xlabel("R (Session 1)")
plt.ylabel("R (Session 2)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# # --- Save high-res image using sanitized title ---
# save_path = r"C:\Users\pkell\OneDrive\Pictures\Rel Match Paper Figures"
# safe_title = title.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")
# fig.savefig(f"{save_path}\\{safe_title}.png", dpi=300, bbox_inches='tight')
plt.show()

✅ Group-level R correlation across sessions: r = 0.9982, p = 7.032494337000244e-109
Number of matched channel pairs: 90


In [422]:
p_val

0.0

In [314]:
## Group Level Condition Correlation - Functional Connectivity

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Label sessions ---
df1['session'] = 1
df2['session'] = 2

# --- Get conditions ---
conditions = sorted(set(df1['condition'].unique()).intersection(df2['condition'].unique()))

# --- Prepare for plotting ---
plt.figure(figsize=(10, 5))

for idx, cond in enumerate(conditions):
    df1_cond = df1[df1['condition'] == cond]
    df2_cond = df2[df2['condition'] == cond]

    merged = pd.merge(
        df1_cond, df2_cond,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    ).dropna(subset=['R_s1', 'R_s2'])

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough data for condition {cond}, skipping.")
        continue

    # Compute Pearson correlation of R values
    r_val, p_val = pearsonr(merged['R_s1'], merged['R_s2'])

    print(f"✅ Condition {cond}: r = {r_val:.4f}, p = {p_val:.4g}, n = {len(merged)}")

    # Plot
    plt.subplot(1, len(conditions), idx + 1)
    sns.scatterplot(x=merged['R_s1'], y=merged['R_s2'], s=60, edgecolor='black',
                    color='mediumorchid' if cond == 1 else 'steelblue')
    plt.plot([merged['R_s1'].min(), merged['R_s1'].max()],
             [merged['R_s1'].min(), merged['R_s1'].max()],
             linestyle='--', color='red')
    plt.xlabel("R (Session 1)")
    plt.ylabel("R (Session 2)")
    plt.title(f"Condition {cond}\nr = {r_val:.2f}, p = {p_val:.3g}")
    plt.grid(True, linestyle='--', alpha=0.5)

plt.suptitle("Session-to-Session Functional Connectivity Stability\n(Group-Level R Values by Condition)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


✅ Condition 1: r = 0.9762, p = 0, n = 2070
✅ Condition 2: r = 0.9742, p = 0, n = 2070


In [458]:
## Group Level Task vs Rest Correlation - Functional Connectivity

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Labels ---
df1['session'] = 1
df2['session'] = 2

# --- Merge on Channel Pair Info ---
merged = pd.merge(
    df1, df2,
    on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
    suffixes=('_s1', '_s2')
).dropna(subset=['R_s1', 'R_s2'])

# --- Compute Group-Level Correlation of R Values ---
r_val, p_val = pearsonr(merged['R_s1'], merged['R_s2'])

print(f"✅ Group-level R correlation across sessions: r = {r_val:.4f}, p = {p_val:.4g}")
print(f"Number of matched channel pairs: {len(merged)}")

# --- Plotting ---
plt.figure(figsize=(6, 4))

# Red dots with black edge
sns.scatterplot(x=merged['R_s1'], y=merged['R_s2'], s=60, color='red', edgecolor='black')

# Black identity line
plt.plot(
    [merged['R_s1'].min(), merged['R_s1'].max()],
    [merged['R_s1'].min(), merged['R_s1'].max()],
    linestyle='--', color='black'
)

plt.title(f"Session-to-Session Functional Connectivity (Channel)\nr = {r_val:.2f}, p = {p_val:.3g}", fontsize=12)
plt.xlabel("R (Session 1)")
plt.ylabel("R (Session 2)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()



✅ Group-level R correlation across sessions: r = 0.9770, p = 0
Number of matched channel pairs: 2070


In [218]:
## Group Level Condition level Correlation of VARIANCE - Functional Connectivity

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_ROI_AGG_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_ROI_AGG_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Label sessions ---
df1['session'] = 1
df2['session'] = 2

# --- Get conditions ---
conditions = sorted(set(df1['condition'].unique()).intersection(df2['condition'].unique()))

# --- Prepare for plotting ---
plt.figure(figsize=(10, 5))

for idx, cond in enumerate(conditions):
    df1_cond = df1[df1['condition'] == cond]
    df2_cond = df2[df2['condition'] == cond]

    merged = pd.merge(
        df1_cond, df2_cond,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    ).dropna(subset=['R_s1', 'R_s2'])

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough data for condition {cond}, skipping.")
        continue

    # Compute variances (R²)
    var_s1 = merged['R_s1'] ** 2
    var_s2 = merged['R_s2'] ** 2

    # Pearson correlation of variances
    r_var, p_var = pearsonr(var_s1, var_s2)

    print(f"✅ Condition {cond}: r = {r_var:.4f}, p = {p_var:.4g}, n = {len(var_s1)}")

    # Plot
    plt.subplot(1, len(conditions), idx+1)
    sns.scatterplot(x=var_s1, y=var_s2, s=60, edgecolor='black', color='mediumorchid' if cond == 1 else 'steelblue')
    plt.plot([var_s1.min(), var_s1.max()], [var_s1.min(), var_s1.max()], linestyle='--', color='red')
    plt.xlabel("Variance in Session 1 (R²)")
    plt.ylabel("Variance in Session 2 (R²)")
    plt.title(f"Condition {cond}\nr = {r_var:.2f}, p = {p_var:.3g}")
    plt.grid(True, linestyle='--', alpha=0.5)

plt.suptitle("Stability of Functional Connectivity Variance\nby Condition", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


✅ Condition 1: r = 0.9813, p = 8.351e-65, n = 90
✅ Condition 2: r = 0.9815, p = 6.337e-65, n = 90


In [288]:
## Group Level Task v Rest Correlation of VARIANCE- Functional Connectivity
import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Label ---
df1['session'] = 1
df2['session'] = 2

# --- Merge on Channel Pair Info ---
merged = pd.merge(
    df1, df2,
    on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
    suffixes=('_s1', '_s2')
).dropna(subset=['R_s1', 'R_s2'])

# --- Compute Variance in Functional Connectivity Across Channel Pairs ---
# You can also group by ROI or connection type here if needed
var_s1 = merged['R_s1'] ** 2
var_s2 = merged['R_s2'] ** 2

# --- Compute Pearson r of Variance Pattern Across Sessions ---
r_var, p_var = pearsonr(var_s1, var_s2)

print(f"✅ Variance pattern correlation across sessions: r = {r_var:.4f}, p = {p_var:.4g}")
print(f"Number of matched channel pairs: {len(var_s1)}")

# --- Optional: Scatter Plot of Variance Stability ---
plt.figure(figsize=(6, 4))
sns.scatterplot(x=var_s1, y=var_s2, s=60, color='mediumseagreen', edgecolor='black')
plt.plot([var_s1.min(), var_s1.max()],
         [var_s1.min(), var_s1.max()],
         linestyle='--', color='red')

plt.title(f"Session-to-Session Stability of Variance Functional Connectivity (Task vs Rest)\nr = {r_val:.2f}, p = {p_val:.3g}", fontsize=12)
plt.xlabel("Variance in Session 1 (R²)")
plt.ylabel("Variance in Session 2 (R²)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Variance pattern correlation across sessions: r = 0.9777, p = 0
Number of matched channel pairs: 2070


In [236]:
## Subject Level Correlation of Variance - Condition - Functional Connectivity

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_ROI_AGG_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_ROI_AGG_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add session indicator ---
df1['session'] = 1
df2['session'] = 2

# --- Get common subjects ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine for iteration ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Per-participant per-condition variance stability results ---
results = []

for subject in combined['subject'].unique():
    for cond in combined['condition'].unique():
        subj_data = combined[(combined['subject'] == subject) & (combined['condition'] == cond)]

        session1 = subj_data[subj_data['session'] == 1]
        session2 = subj_data[subj_data['session'] == 2]

        merged = pd.merge(
            session1,
            session2,
            on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
            suffixes=('_s1', '_s2')
        ).dropna(subset=['R_s1', 'R_s2'])

        if merged.shape[0] < 3:
            print(f"⚠️ Skipping {subject} condition {cond}: not enough matched pairs.")
            continue

        var_s1 = merged['R_s1'] ** 2
        var_s2 = merged['R_s2'] ** 2

        r_var, p_var = pearsonr(var_s1, var_s2)

        results.append({
            'subject': subject,
            'condition': cond,
            'r_var': r_var,
            'p_var': p_var,
            'n_pairs': len(var_s1)
        })

# --- Create results dataframe ---
variance_df = pd.DataFrame(results)
print(variance_df)

# --- Clean Bar Plot: Grouped by Subject and Condition ---
plt.figure(figsize=(12, 6))
sns.set(style='whitegrid')

# Sort subjects consistently
variance_df['subject'] = variance_df['subject'].astype(str)
variance_df = variance_df.sort_values(by='subject')

# Bar plot with side-by-side bars per subject
ax = sns.barplot(
    data=variance_df,
    x='subject',
    y='r_var',
    hue='condition',
    palette='Set2',
    edgecolor='black'
)

# Add text labels above bars
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", label_type='edge', padding=3, fontsize=9)

# Clean axis and legend
plt.title("Participant-Level Variance Stability by Condition\n(R² Correlation: Session 1 vs Session 2)", fontsize=14)
plt.xlabel("Subject", fontsize=12)
plt.ylabel("Pearson r (Variance)", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.3, 1.05)
plt.legend(title='Condition', title_fontsize=11, fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()


   subject  condition     r_var          p_var  n_pairs
0   pid001          1  0.942648   1.700854e-48      100
1   pid001          2  0.945931   1.024710e-49      100
2   pid002          1  0.982358   3.598704e-73      100
3   pid002          2  0.979854   2.261731e-70      100
4   pid003          1  0.927570   1.093492e-43      100
5   pid003          2  0.958575   2.980681e-55      100
6   pid004          1  0.953119   1.122962e-52      100
7   pid004          2  0.928381   6.419829e-44      100
8   pid005          1  0.979570   4.456958e-70      100
9   pid005          2  0.961884   5.454364e-57      100
10  pid006          1  0.978504   5.254553e-69      100
11  pid006          2  0.980978   1.394912e-71      100
12  pid007          1  0.977982   1.682461e-68      100
13  pid007          2  0.970821   1.392772e-62      100
14  pid008          1  0.981947   1.099118e-72      100
15  pid008          2  0.969847   6.806516e-62      100
16  pid009          1  0.974892   9.729812e-66  

In [232]:
## Subject Level Correlation of Variance - Functional Connectivity

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_ROI_AGG_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_ROI_AGG_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add session indicator ---
df1['session'] = 1
df2['session'] = 2

# --- Get common subjects ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine for iteration ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Per-participant variance stability results ---
results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    merged = pd.merge(
        session1,
        session2,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    ).dropna(subset=['R_s1', 'R_s2'])

    if merged.shape[0] < 3:
        print(f"⚠️ Skipping {subject}: not enough matched pairs.")
        continue

    var_s1 = merged['R_s1'] ** 2
    var_s2 = merged['R_s2'] ** 2

    r_var, p_var = pearsonr(var_s1, var_s2)

    results.append({
        'subject': subject,
        'r_var': r_var,
        'p_var': p_var,
        'n_pairs': len(var_s1)
    })

# --- Create results dataframe ---
variance_df = pd.DataFrame(results)
print(variance_df)

# --- Plotting ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=variance_df, x='subject', y='r_var', edgecolor='black', palette='viridis')

for i, row in variance_df.iterrows():
    ax.text(i, row['r_var'] + 0.02, f"{row['r_var']:.2f}", ha='center', va='bottom', fontsize=9)

plt.title("Participant-Level Variance Stability\n(R² Correlation: Session 1 vs Session 2)", fontsize=14)
plt.xlabel("Subject")
plt.ylabel("Pearson r (Variance Session 1 vs 2)")
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.3, 1.05)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


   subject     r_var          p_var  n_pairs
0   pid001  0.947884   1.771289e-50      100
1   pid002  0.982605   1.813192e-73      100
2   pid003  0.946454   6.442440e-50      100
3   pid004  0.944639   3.157584e-49      100
4   pid005  0.974034   4.949559e-65      100
5   pid006  0.980913   1.642246e-71      100
6   pid007  0.974492   2.090830e-65      100
7   pid008  0.978313   8.062222e-69      100
8   pid009  0.975199   5.370092e-66      100
9   pid010  0.982998   5.960516e-74      100
10  pid011  0.980325   7.168896e-71      100
11  pid012  0.980526   4.353510e-71      100
12  pid013  0.954634   2.327072e-53      100
13  pid014  0.990753   7.864552e-87      100
14  pid015  0.969802   7.306206e-62      100
15  pid016  0.881059   1.247081e-33      100
16  pid017  0.990157   1.655046e-85      100
17  pid018  0.659516   8.557842e-14      100
18  pid019  0.993966   6.953122e-96      100
19  pid020  0.974741   1.302873e-65      100
20  pid021  0.976896   1.734778e-67      100
21  pid022

C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\2962233279.py:69: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=variance_df, x='subject', y='r_var', edgecolor='black', palette='viridis')


In [159]:
print(np.mean(variance_df['r_var']))

0.8172641430080828


In [161]:
## Subject Level Variance permutation Test

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Common Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Permutation Parameters ---
n_iter = 1000
permutation_results = []

# --- Run Permutation Test Per Subject ---
for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Merge on the full connection pair (origin + destination)
    merged = pd.merge(
        session1,
        session2,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    ).dropna(subset=['R_s1', 'R_s2'])

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched pairs for {subject}. Skipping.")
        continue

    # Compute variance (R²) vectors
    var_s1 = merged['R_s1'] ** 2
    var_s2 = merged['R_s2'] ** 2

    # True Pearson correlation of variances
    true_r, _ = pearsonr(var_s1, var_s2)
    perm_r_vals = []

    # Permutation test
    for _ in range(n_iter):
        var_s2_perm = np.random.permutation(var_s2)
        r, _ = pearsonr(var_s1, var_s2_perm)
        perm_r_vals.append(r)

    # Compute p-value
    p_value = np.mean(np.array(perm_r_vals) >= true_r)

    permutation_results.append({
        'subject': subject,
        'true_r': true_r,
        'perm_r_vals': perm_r_vals,
        'p_value': p_value
    })

# --- Create DataFrame ---
perm_df = pd.DataFrame(permutation_results)

# --- Plotting ---
plt.figure(figsize=(12, 6))

for _, row in perm_df.iterrows():
    sns.kdeplot(row['perm_r_vals'], fill=True, alpha=0.2, label=f"{row['subject']} permuted")
    plt.axvline(row['true_r'], linestyle='--', linewidth=2, alpha=0.8, label=f"{row['subject']} true r")

plt.title("Within-Subject Permutation Distributions vs. True r Values (Variance Stability)")
plt.xlabel("Pearson r (Variance: Session 1 vs Session 2)")
plt.ylabel("Density")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# --- Print Summary Table ---
print(perm_df[['subject', 'true_r', 'p_value']])


✅ Found 26 subjects in both sessions.
   subject    true_r  p_value
0   pid001  0.764670      0.0
1   pid002  0.885733      0.0
2   pid003  0.845104      0.0
3   pid004  0.860340      0.0
4   pid005  0.791033      0.0
5   pid006  0.833246      0.0
6   pid007  0.893535      0.0
7   pid008  0.854492      0.0
8   pid009  0.736290      0.0
9   pid010  0.864959      0.0
10  pid011  0.847293      0.0
11  pid012  0.791251      0.0
12  pid013  0.754872      0.0
13  pid014  0.882637      0.0
14  pid015  0.817081      0.0
15  pid016  0.718964      0.0
16  pid017  0.956806      0.0
17  pid018  0.373516      0.0
18  pid019  0.837143      0.0
19  pid020  0.814773      0.0
20  pid021  0.884368      0.0
21  pid022  0.847032      0.0
22  pid023  0.895323      0.0
23  pid024  0.874825      0.0
24  pid025  0.705571      0.0
25  pid026  0.918011      0.0


In [310]:
## TASK VS REST - Channel Level Correlation Code - SUBJECT LEVEL 
import pandas as pd
from scipy.stats import pearsonr

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Group-level Pearson correlation across subjects ---
correlation_results = []

for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]
    
    for (src, det), chan_data in cond_data.groupby(['source', 'detector']):
        session1 = chan_data[chan_data['session'] == 1].sort_values('subject')
        session2 = chan_data[chan_data['session'] == 2].sort_values('subject')

        # Match subjects
        common_subs = set(session1['subject']).intersection(set(session2['subject']))
        session1 = session1[session1['subject'].isin(common_subs)].sort_values('subject')
        session2 = session2[session2['subject'].isin(common_subs)].sort_values('subject')

        if len(session1) > 1:
            try:
                r, p = pearsonr(session1['tstat'].values, session2['tstat'].values)
                correlation_results.append({
                    'condition': condition,
                    'channel': f"S{src}_D{det}",
                    'pearson_r': r,
                    'p_value': p,
                    'n_subjects': len(session1)
                })
            except Exception as e:
                print(f"❌ Error computing Pearson r for {condition}, S{src}_D{det}: {e}")

# --- Final Output ---
cor_df = pd.DataFrame(correlation_results)
print(cor_df)
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# --- Step 1: Add significance flag and color ---
cor_df['significant'] = cor_df['p_value'] < 0.05
cor_df['bar_color'] = cor_df['significant'].map({True: 'seagreen', False: 'gray'})

# --- Step 2: Bootstrap CI for each (channel, condition) pair ---
boot_ci = []

for i, row in cor_df.iterrows():
    cond = row['condition']
    chan = row['channel']

    # Get matched data for this condition/channel
    cond_data = combined[
        (combined['cond'] == cond) &
        (combined['source'] == int(chan.split('_')[0][1:])) &
        (combined['detector'] == int(chan.split('_')[1][1:]))
    ]

    session1 = cond_data[cond_data['session'] == 1].sort_values('subject')
    session2 = cond_data[cond_data['session'] == 2].sort_values('subject')

    common_subjects = set(session1['subject']).intersection(set(session2['subject']))
    session1 = session1[session1['subject'].isin(common_subjects)].sort_values('subject')
    session2 = session2[session2['subject'].isin(common_subjects)].sort_values('subject')

    r_boot = []
    if len(session1) > 3:
        for _ in range(1000):
            idx = np.random.choice(len(session1), size=len(session1), replace=True)
            try:
                r, _ = pearsonr(session1['tstat'].values[idx], session2['tstat'].values[idx])
                r_boot.append(r)
            except:
                continue

        lower = np.percentile(r_boot, 2.5)
        upper = np.percentile(r_boot, 97.5)
    else:
        lower, upper = np.nan, np.nan

    boot_ci.append((lower, upper))

cor_df['ci_lower'], cor_df['ci_upper'] = zip(*boot_ci)
cor_df['yerr_lower'] = cor_df['pearson_r'] - cor_df['ci_lower']
cor_df['yerr_upper'] = cor_df['ci_upper'] - cor_df['pearson_r']

# --- Step 3: Plot with CIs and color coding ---
plt.figure(figsize=(14, 6))

# Plot each condition separately for better grouping
for condition in cor_df['condition'].unique():
    subset = cor_df[cor_df['condition'] == condition]
    x_pos = np.arange(len(subset))
    bar = plt.bar(
        x=x_pos,
        height=subset['pearson_r'],
        yerr=[subset['yerr_lower'], subset['yerr_upper']],
        capsize=4,
        color=subset['bar_color'],
        edgecolor='black',
        label=f'Condition {condition}'
    )

    for i, (x, r) in enumerate(zip(x_pos, subset['pearson_r'])):
        plt.text(x, r + 0.02, f"{r:.2f}", ha='center', va='bottom', fontsize=9)

    # Fix x-ticks
    plt.xticks(ticks=x_pos, labels=subset['channel'], rotation=90)

plt.axhline(0, color='gray', linestyle='--')
plt.ylabel('Pearson r')
plt.title('Subject-Level Session-to-Session Correlation by Channel and Condition')
plt.ylim(-0.2, 1.1)
plt.legend()
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# --- Compute Average Pearson r for Each Condition ---
average_corr = (
    cor_df
    .groupby('condition')['pearson_r']
    .mean()
    .reset_index()
    .rename(columns={'pearson_r': 'avg_pearson_r'})
)

print("📊 Average Pearson r by Condition:")
print(average_corr)





✅ Found 26 subjects in both sessions.
   condition  channel  pearson_r   p_value  n_subjects
0         x1    S1_D1   0.319864  0.111170          26
1         x1    S1_D2   0.244033  0.229596          26
2         x1    S2_D1   0.227862  0.262911          26
3         x1    S2_D2   0.263481  0.193415          26
4         x1    S2_D3  -0.263349  0.193646          26
5         x1    S3_D1  -0.257588  0.203936          26
6         x1    S3_D3   0.007899  0.969449          26
7         x1    S4_D2   0.216162  0.288867          26
8         x1    S4_D3   0.175798  0.390324          26
9         x1    S4_D4   0.468168  0.015863          26
10        x1    S5_D3   0.206075  0.312498          26
11        x1    S5_D4   0.005683  0.978019          26
12        x1   S5_D16   0.463269  0.017150          26
13        x1    S6_D5   0.598358  0.001242          26
14        x1    S6_D6   0.544711  0.004011          26
15        x1    S7_D5   0.425222  0.030345          26
16        x1    S7_D6   0.2

In [308]:
## TASK VS REST - Channel Level Variance Correlation Code - SUBJECT LEVEL 

import pandas as pd
pd.set_option('display.max_rows', None)
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Group-level Variance Correlation across subjects ---
variance_results = []

for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]
    
    for (src, det), chan_data in cond_data.groupby(['source', 'detector']):
        session1 = chan_data[chan_data['session'] == 1].sort_values('subject')
        session2 = chan_data[chan_data['session'] == 2].sort_values('subject')

        common_subs = set(session1['subject']).intersection(set(session2['subject']))
        session1 = session1[session1['subject'].isin(common_subs)].sort_values('subject')
        session2 = session2[session2['subject'].isin(common_subs)].sort_values('subject')

        if len(session1) > 1:
            try:
                var1 = session1['tstat'].values ** 2
                var2 = session2['tstat'].values ** 2
                r, p = pearsonr(var1, var2)
                variance_results.append({
                    'condition': condition,
                    'channel': f"S{src}_D{det}",
                    'pearson_r_variance': r,
                    'p_value': p,
                    'n_subjects': len(session1)
                })
            except Exception as e:
                print(f"❌ Error computing variance correlation for {condition}, S{src}_D{det}: {e}")

# --- Final Output ---
var_df = pd.DataFrame(variance_results)
print(var_df)
# --- Add significance flag and color ---
var_df['significant'] = var_df['p_value'] < 0.05
var_df['bar_color'] = var_df['significant'].map({True: 'seagreen', False: 'gray'})

# --- Plot with CIs and color coding ---
plt.figure(figsize=(14, 6))

for condition in var_df['condition'].unique():
    subset = var_df[var_df['condition'] == condition]
    x_pos = np.arange(len(subset))

    bar = plt.bar(
        x=x_pos,
        height=subset['pearson_r_variance'],
        color=subset['bar_color'],
        edgecolor='black',
        label=f'Condition {condition}'
    )

    for i, (x, r) in enumerate(zip(x_pos, subset['pearson_r_variance'])):
        plt.text(x, r + 0.02, f"{r:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(ticks=x_pos, labels=subset['channel'], rotation=90)

plt.axhline(0, color='gray', linestyle='--')
plt.ylabel('Pearson r (t²)')
plt.title('Subject-Level Session-to-Session Variance Correlation by Channel and Condition')
plt.ylim(-0.2, 1.1)
plt.legend()
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# --- Compute Average Variance Correlation for Each Condition ---
average_var_corr = (
    var_df
    .groupby('condition')['pearson_r_variance']
    .mean()
    .reset_index()
    .rename(columns={'pearson_r_variance': 'avg_pearson_r_variance'})
)

print("📊 Average Pearson r (t²) by Condition:")
print(average_var_corr)


   condition  channel  pearson_r_variance   p_value  n_subjects
0         x1    S1_D1            0.180537  0.377467          26
1         x1    S1_D2           -0.048460  0.814145          26
2         x1    S2_D1            0.182850  0.371284          26
3         x1    S2_D2            0.091929  0.655134          26
4         x1    S2_D3            0.185973  0.363026          26
5         x1    S3_D1           -0.025782  0.900509          26
6         x1    S3_D3           -0.140721  0.492919          26
7         x1    S4_D2            0.093661  0.649040          26
8         x1    S4_D3            0.063979  0.756178          26
9         x1    S4_D4            0.178532  0.382878          26
10        x1    S5_D3            0.418102  0.033543          26
11        x1    S5_D4           -0.216456  0.288196          26
12        x1   S5_D16            0.223610  0.272162          26
13        x1    S6_D5            0.655594  0.000277          26
14        x1    S6_D6            0.41635

In [110]:
plt.figure(figsize=(12, 6))
sns.barplot(data=cor_df, x='channel', y='pearson_r', hue='condition', dodge=True)

plt.xticks(rotation=90)
plt.ylabel('Pearson r')
plt.title('Session-to-Session Correlation by Channel and Condition')
plt.legend(title='Condition')
plt.tight_layout()
plt.show()


In [112]:
import matplotlib.pyplot as plt

# --- Bar Plot of Average Pearson r by Condition ---
plt.figure(figsize=(6, 4))
plt.bar(average_corr['condition'].astype(str), average_corr['avg_pearson_r'])

plt.xlabel('Condition')
plt.ylabel('Average Pearson r')
plt.title('Average Pearson Correlation by Condition')
plt.ylim(-.2, 1)  # since Pearson r ranges from -1 to 1, but likely positive here
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

KeyError: 'avg_pearson_r'

In [296]:
## Channel Level Variance Correlation
import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Channel-level Variance Correlation Across Subjects ---
correlation_results = []

for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]
    
    for (src, det), chan_data in cond_data.groupby(['source', 'detector']):
        session1 = chan_data[chan_data['session'] == 1].sort_values('subject')
        session2 = chan_data[chan_data['session'] == 2].sort_values('subject')

        # Match subjects
        common_subs = set(session1['subject']).intersection(set(session2['subject']))
        session1 = session1[session1['subject'].isin(common_subs)].sort_values('subject')
        session2 = session2[session2['subject'].isin(common_subs)].sort_values('subject')

        if len(session1) > 1:
            try:
                # Use squared t-values (proxy for variance)
                var_s1 = session1['tstat'].values ** 2
                var_s2 = session2['tstat'].values ** 2

                r, p = pearsonr(var_s1, var_s2)
                correlation_results.append({
                    'condition': condition,
                    'channel': f"S{src}_D{det}",
                    'pearson_r': r,
                    'p_value': p,
                    'n_subjects': len(session1)
                })
            except Exception as e:
                print(f"❌ Error computing variance r for {condition}, S{src}_D{det}: {e}")

# --- Final Output ---
cor_df = pd.DataFrame(correlation_results)
print(cor_df)

# --- Plot 1: Channel-level Pearson r split by condition ---
plt.figure(figsize=(12, 6))
sns.barplot(data=cor_df, x='channel', y='pearson_r', hue='condition', edgecolor='black')
plt.xticks(rotation=90)
plt.title("Channel-Level Variance Stability (r) by Condition")
plt.ylabel("Pearson r (t² Session 1 vs Session 2)")
plt.xlabel("Channel")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# --- Plot 2: Channel-wise Pearson r across all channels (not split by condition) ---
plt.figure(figsize=(12, 6))
sns.barplot(data=cor_df, x='channel', y='pearson_r', edgecolor='black', color='skyblue')
plt.xticks(rotation=90)
plt.title("Channel-Level Variance Stability (All Conditions Combined)")
plt.ylabel("Pearson r (t² Session 1 vs Session 2)")
plt.xlabel("Channel")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


# --- Print Mean Variance Stability per Condition ---
average_corr = (
    cor_df
    .groupby('condition')['pearson_r']
    .mean()
    .reset_index()
    .rename(columns={'pearson_r': 'avg_variance_r'})
)
print("📊 Average Variance Stability (r) by Condition:")
print(average_corr)


✅ Found 26 subjects in both sessions.
   condition  channel  pearson_r   p_value  n_subjects
0         x1    S1_D1   0.180537  0.377467          26
1         x1    S1_D2  -0.048460  0.814145          26
2         x1    S2_D1   0.182850  0.371284          26
3         x1    S2_D2   0.091929  0.655134          26
4         x1    S2_D3   0.185973  0.363026          26
..       ...      ...        ...       ...         ...
87        x2  S16_D12  -0.203698  0.318235          26
88        x2  S16_D13  -0.011909  0.953957          26
89        x2  S16_D14   0.200492  0.326075          26
90        x2  S16_D15   0.034863  0.865738          26
91        x2  S16_D23   0.126091  0.539362          26

[92 rows x 5 columns]
📊 Average Variance Stability (r) by Condition:
  condition  avg_variance_r
0        x1        0.105588
1        x2        0.179729


In [396]:
## Participant Session to Session Correlations - TASK VS REST  

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_ROI_AGG_subject_glm_result_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_ROI_AGG_subject_glm_result_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Drop unnecessary columns if they exist ---
df1 = df1.drop(columns=['contact_info', 'remarks'], errors='ignore')
df2 = df2.drop(columns=['contact_info', 'remarks'], errors='ignore')

# --- Drop only rows with NaNs ---
df1 = df1.dropna(how='any')
df2 = df2.dropna(how='any')


# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r for Each Participant Across Channels ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]
    
    session1 = subj_data[subj_data['session'] == 1].copy()
    session2 = subj_data[subj_data['session'] == 2].copy()

    # Match channels by source and detector
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector', 'cond'],  # Match on channel and condition
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough channels to compute correlation for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_channels': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

import matplotlib.pyplot as plt
import seaborn as sns

# --- Bar Plot of Session-to-Session r for Each Participant ---
plt.figure(figsize=(12, 6))
sns.set(style="whitegrid", font_scale=1.2)

# Create barplot
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='Blues_d')

# Add r value text above each bar
for index, row in subject_corr_df.iterrows():
    ax.text(index, row['pearson_r'] + 0.01, f"{row['pearson_r']:.2f}", ha='center', va='bottom', fontsize=10)

# Final plot formatting
plt.xticks(rotation=45, ha='right')
plt.ylabel("Pearson r (Session 1 vs Session 2)")
plt.xlabel("Participant")
plt.title("Participant Level Test-Retest Correlation (ROI)")
plt.ylim(-.6, 1)  # Adjust as needed based on r value range
plt.tight_layout()
plt.show()



✅ Found 23 subjects in both sessions.
   subject  pearson_r  n_channels
0   pid001   0.206518           9
1   pid002   0.327740          10
2   pid003   0.347933          10
3   pid005   0.444450          10
4   pid006  -0.523050           8
5   pid007   0.722128           6
6   pid008   0.863505          10
7   pid009   0.775865           7
8   pid010   0.421798          10
9   pid011   0.790687           9
10  pid012  -0.138084           4
11  pid013   0.442626           9
12  pid014   0.421439          10
13  pid016  -0.255521           8
14  pid017  -0.395235           8
15  pid018   0.012844          10
16  pid020   0.712728          10
17  pid021   0.751693           6
18  pid022   0.553405           7
19  pid023   0.188407           8
20  pid024   0.905505           7
21  pid025   0.816631           9
22  pid026   0.559036          10


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\3491504605.py:85: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='Blues_d')


In [402]:
## Subject Level Variance Correlation Task vs Rest 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_ROI_AGG_subject_glm_result_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_ROI_AGG_subject_glm_result_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Drop unnecessary columns if they exist ---
df1 = df1.drop(columns=['contact_info', 'remarks'], errors='ignore')
df2 = df2.drop(columns=['contact_info', 'remarks'], errors='ignore')

# --- Drop only rows with NaNs ---
df1 = df1.dropna(how='any')
df2 = df2.dropna(how='any')

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r on t² for Each Participant Across Channels ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]
    
    session1 = subj_data[subj_data['session'] == 1].copy()
    session2 = subj_data[subj_data['session'] == 2].copy()

    # Match on channel and condition
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector', 'cond'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}. Skipping.")
        continue

    try:
        t2_s1 = merged['tstat_s1']**2
        t2_s2 = merged['tstat_s2']**2
        r, p = pearsonr(t2_s1, t2_s2)
        participant_results.append({
            'subject': subject,
            'pearson_r_variance': r,
            'n_channels': len(t2_s1)
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
var_stab_df = pd.DataFrame(participant_results)
print(var_stab_df)

# --- Plot Histogram of Variance r Values ---
plt.figure(figsize=(8, 5))
plt.hist(var_stab_df['pearson_r_variance'], bins=10, edgecolor='black', color='teal')
plt.title('Participant-Level Variance Stability (t² Correlation)')
plt.xlabel('Pearson r (t² Session 1 vs Session 2)')
plt.ylabel('Number of Participants')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# --- Bar Plot of Variance Stability per Participant ---
import seaborn as sns

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=var_stab_df, x='subject', y='pearson_r_variance', palette='coolwarm', edgecolor='black')

# Add text labels above each bar
for i, row in var_stab_df.iterrows():
    ax.text(
        i,
        row['pearson_r_variance'] + 0.02,
        f"{row['pearson_r_variance']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Variance Stability (ROI)', fontsize=14)
plt.xlabel('Participant')
plt.ylabel('Pearson r (t² Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.6, 1.1)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()



✅ Found 23 subjects in both sessions.
   subject  pearson_r_variance  n_channels
0   pid001            0.259190           9
1   pid002           -0.168064          10
2   pid003            0.384782          10
3   pid005            0.606556          10
4   pid006           -0.465195           8
5   pid007            0.756680           6
6   pid008            0.703114          10
7   pid009            0.144024           7
8   pid010           -0.225039          10
9   pid011            0.866318           9
10  pid012           -0.316364           4
11  pid013           -0.129830           9
12  pid014            0.041141          10
13  pid016           -0.082294           8
14  pid017           -0.495111           8
15  pid018           -0.090791          10
16  pid020            0.597585          10
17  pid021            0.450726           6
18  pid022            0.732506           7
19  pid023            0.087961           8
20  pid024            0.103531           7
21  pid025      

C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\3249592614.py:92: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=var_stab_df, x='subject', y='pearson_r_variance', palette='coolwarm', edgecolor='black')


In [121]:
## Subject Level Variance Task vs Rest Permutation

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Common Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Data ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Permutation Test ---
n_iter = 1000
perm_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    s1 = subj_data[subj_data['session'] == 1]
    s2 = subj_data[subj_data['session'] == 2]

    # Match on channel and condition
    merged = pd.merge(
        s1,
        s2,
        on=['source', 'detector', 'cond'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}. Skipping.")
        continue

    try:
        t2_s1 = merged['tstat_s1']**2
        t2_s2 = merged['tstat_s2']**2

        true_r, _ = pearsonr(t2_s1, t2_s2)
        perm_r_vals = []

        for _ in range(n_iter):
            permuted = np.random.permutation(t2_s2)
            r, _ = pearsonr(t2_s1, permuted)
            perm_r_vals.append(r)

        p_value = np.mean(np.array(perm_r_vals) >= true_r)

        perm_results.append({
            'subject': subject,
            'true_r': true_r,
            'p_value': p_value,
            'perm_r_vals': perm_r_vals
        })

    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Format Output ---
perm_df = pd.DataFrame(perm_results)

# --- Plot All Subjects’ Permuted Distributions and True r Lines ---
plt.figure(figsize=(12, 6))
for _, row in perm_df.iterrows():
    sns.kdeplot(row['perm_r_vals'], fill=True, alpha=0.25, label=f"{row['subject']} permuted")
    plt.axvline(row['true_r'], linestyle='--', linewidth=1.5, alpha=0.8, label=f"{row['subject']} true r")

plt.title("Permutation Test: Variance Stability by Participant")
plt.xlabel("Pearson r (t² Session 1 vs Session 2)")
plt.ylabel("Density")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

# --- Show Summary Table ---
print(perm_df[['subject', 'true_r', 'p_value']])


✅ Found 26 subjects in both sessions.
   subject    true_r  p_value
0   pid001  0.208258    0.091
1   pid002  0.179553    0.113
2   pid003  0.640750    0.000
3   pid004  0.733088    0.000
4   pid005  0.300130    0.031
5   pid006 -0.096367    0.743
6   pid007  0.507570    0.000
7   pid008  0.277048    0.049
8   pid009  0.131763    0.193
9   pid010 -0.063227    0.619
10  pid011  0.481062    0.003
11  pid012  0.167081    0.138
12  pid013 -0.004750    0.465
13  pid014  0.051026    0.316
14  pid015  0.521748    0.005
15  pid016  0.164426    0.114
16  pid017 -0.254639    0.977
17  pid018  0.068038    0.278
18  pid019  0.198586    0.079
19  pid020  0.354748    0.009
20  pid021  0.736696    0.000
21  pid022  0.220573    0.064
22  pid023  0.116671    0.247
23  pid024  0.106076    0.210
24  pid025  0.610746    0.000
25  pid026  0.372536    0.008


In [576]:
## Subject Level Condition Correlation

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r for Each Subject × Condition Across Channels ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    for condition in subj_data['cond'].unique():
        cond_data = subj_data[subj_data['cond'] == condition]

        session1 = cond_data[cond_data['session'] == 1]
        session2 = cond_data[cond_data['session'] == 2]

        merged = pd.merge(
            session1,
            session2,
            on=['source', 'detector'],  # Match on channel
            suffixes=('_s1', '_s2')
        )

        if merged.shape[0] < 3:
            print(f"⚠️ Not enough channels for {subject}, {condition}. Skipping.")
            continue

        try:
            r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
            participant_results.append({
                'subject': subject,
                'condition': condition,
                'pearson_r': r,
                'n_channels': merged.shape[0]
            })
        except Exception as e:
            print(f"❌ Error for {subject}, {condition}: {e}")

# --- Results DataFrame ---
subject_cond_corr_df = pd.DataFrame(participant_results)
print(subject_cond_corr_df)

# --- Plot Histograms per Condition ---
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(data=subject_cond_corr_df, x='pearson_r', hue='condition', bins=10, kde=True, edgecolor='black')
plt.title('Participant-Level Test–Retest Correlations by Condition')
plt.xlabel('Pearson r (Session 1 vs Session 2)')
plt.ylabel('Number of Participants')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\pkell\\OneDrive\\Documents\\MATLAB\\FINAL_subject_glm_result_session1.csv'

In [125]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Pivot the dataframe so each row is a subject, and columns are conditions ---
pivot_df = subject_cond_corr_df.pivot(index='subject', columns='condition', values='pearson_r').reset_index()

# --- Convert to long format for plotting ---
long_df = pivot_df.melt(id_vars='subject', var_name='condition', value_name='pearson_r')

# --- Sort subjects for consistent ordering ---
long_df = long_df.sort_values('subject')

# --- Create bar plot ---
plt.figure(figsize=(10, 6))
sns.barplot(data=long_df, x='subject', y='pearson_r', hue='condition', palette='Set2')

# --- Customize the plot ---
plt.title


<function matplotlib.pyplot.title(label: 'str', fontdict: 'dict[str, Any] | None' = None, loc: "Literal['left', 'center', 'right'] | None" = None, pad: 'float | None' = None, *, y: 'float | None' = None, **kwargs) -> 'Text'>

In [127]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Assuming subject_cond_corr_df already exists from earlier code ---
# Columns: ['subject', 'condition', 'pearson_r']

# --- Step 1: Compute average r by condition ---
avg_r_by_condition = (
    subject_cond_corr_df
    .groupby('condition')['pearson_r']
    .mean()
    .reset_index()
    .rename(columns={'pearson_r': 'avg_pearson_r'})
)

print("📊 Average Pearson r by Condition:")
print(avg_r_by_condition)

# --- Step 2: Bar Plot of Averages ---
plt.figure(figsize=(6, 5))
sns.barplot(data=avg_r_by_condition, x='condition', y='avg_pearson_r', palette='Set2')

# --- Customize Plot ---
plt.title('Average Session Correlation by Condition')
plt.ylabel('Average Pearson r')
plt.xlabel('Condition')
plt.ylim(0, .6)
plt.axhline(0, linestyle='--', color='gray', alpha=0.6)
plt.tight_layout()
plt.show()


📊 Average Pearson r by Condition:
  condition  avg_pearson_r
0        x1       0.295144
1        x2       0.351797


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1002506845.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=avg_r_by_condition, x='condition', y='avg_pearson_r', palette='Set2')


In [329]:
## Within-Subject Permutation Test

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_subject_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_subject_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Drop unwanted columns if present ---
df1 = df1.drop(columns=['contact_info', 'remarks'], errors='ignore')
df2 = df2.drop(columns=['contact_info', 'remarks'], errors='ignore')

# --- Drop rows with NaN in 'tstat' column ---
df1 = df1.dropna(subset=['tstat'])
df2 = df2.dropna(subset=['tstat'])

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Permutation Test Parameters ---
n_iter = 1000
permutation_results = []

# --- Run Permutation Test Per Subject and Condition ---
for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]
    
    for condition in subj_data['cond'].unique():
        cond_data = subj_data[subj_data['cond'] == condition]
        s1 = cond_data[cond_data['session'] == 1].sort_values(['source', 'detector'])['tstat'].values
        s2 = cond_data[cond_data['session'] == 2].sort_values(['source', 'detector'])['tstat'].values
        
        if len(s1) < 3 or len(s1) != len(s2):
            print(f"⚠️ Skipping {subject}, {condition}: not enough matching channels.")
            continue

        # True r
        true_r, _ = pearsonr(s1, s2)
        perm_r_vals = []

        # Permutations
        for _ in range(n_iter):
            s2_perm = np.random.permutation(s2)
            r, _ = pearsonr(s1, s2_perm)
            perm_r_vals.append(r)

        permutation_results.append({
            'subject': subject,
            'condition': condition,
            'true_r': true_r,
            'perm_r_vals': perm_r_vals,
            'p_value': np.mean(np.array(perm_r_vals) >= true_r)  # one-tailed
        })

# --- Create DataFrame from Results ---
perm_df = pd.DataFrame(permutation_results)

# --- Plotting ---
plt.figure(figsize=(12, 6))
colors = {'x1': 'skyblue', 'x2': 'salmon'}

for condition in perm_df['condition'].unique():
    subset = perm_df[perm_df['condition'] == condition]

    for _, row in subset.iterrows():
        # Permutation histogram
        sns.kdeplot(row['perm_r_vals'], fill=True, color=colors[condition], alpha=0.25, label=f'{row["subject"]} ({condition}) permuted')

        # Observed r line
        plt.axvline(row['true_r'], color=colors[condition], linestyle='--', linewidth=2, alpha=0.9)

# Formatting
plt.title("Within-Subject Permutation Distributions vs. True r Values")
plt.xlabel("Pearson r")
plt.ylabel("Density")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# Optional: Print p-values for reference
print(perm_df[['subject', 'condition', 'true_r', 'p_value']])


✅ Found 25 subjects in both sessions.
⚠️ Skipping pid001, x1: not enough matching channels.
⚠️ Skipping pid001, x2: not enough matching channels.
⚠️ Skipping pid009, x1: not enough matching channels.
⚠️ Skipping pid009, x2: not enough matching channels.
⚠️ Skipping pid011, x1: not enough matching channels.
⚠️ Skipping pid011, x2: not enough matching channels.
⚠️ Skipping pid012, x1: not enough matching channels.
⚠️ Skipping pid012, x2: not enough matching channels.
⚠️ Skipping pid013, x1: not enough matching channels.
⚠️ Skipping pid013, x2: not enough matching channels.
⚠️ Skipping pid016, x1: not enough matching channels.
⚠️ Skipping pid016, x2: not enough matching channels.
⚠️ Skipping pid017, x1: not enough matching channels.
⚠️ Skipping pid017, x2: not enough matching channels.
⚠️ Skipping pid019, x1: not enough matching channels.
⚠️ Skipping pid019, x2: not enough matching channels.
⚠️ Skipping pid022, x1: not enough matching channels.
⚠️ Skipping pid022, x2: not enough matching 

In [131]:
## Subject Level Condition Variance Stability

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Variance Stability for Each Subject × Condition ---
var_corr_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    for condition in subj_data['cond'].unique():
        cond_data = subj_data[subj_data['cond'] == condition]

        session1 = cond_data[cond_data['session'] == 1]
        session2 = cond_data[cond_data['session'] == 2]

        merged = pd.merge(
            session1,
            session2,
            on=['source', 'detector'],
            suffixes=('_s1', '_s2')
        )

        if merged.shape[0] < 3:
            print(f"⚠️ Not enough channels for {subject}, {condition}. Skipping.")
            continue

        try:
            # Compute t^2 and then correlation of variances
            t2_s1 = merged['tstat_s1'] ** 2
            t2_s2 = merged['tstat_s2'] ** 2
            r, p = pearsonr(t2_s1, t2_s2)

            var_corr_results.append({
                'subject': subject,
                'condition': condition,
                'pearson_r_var': r,
                'n_channels': len(t2_s1)
            })

        except Exception as e:
            print(f"❌ Error for {subject}, {condition}: {e}")

# --- Create DataFrame ---
var_corr_df = pd.DataFrame(var_corr_results)
print(var_corr_df.head())

# --- Plot: Variance Correlation by Condition ---
plt.figure(figsize=(10, 6))
sns.barplot(data=var_corr_df, x='subject', y='pearson_r_var', hue='condition', edgecolor='black')
plt.title("Participant-Level Variance Stability (t² Correlation) by Condition")
plt.xlabel("Subject")
plt.ylabel("Pearson r (t² Session 1 vs Session 2)")
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.3, .7)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


✅ Found 26 subjects in both sessions.
  subject condition  pearson_r_var  n_channels
0  pid001        x1      -0.091917          46
1  pid001        x2       0.143172          46
2  pid002        x1      -0.044658          46
3  pid002        x2       0.328247          46
4  pid003        x1       0.385466          46


In [331]:
## Subject Level Variance Permuation 

## Subject Level Variance Permutation

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_subject_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\ROI_AGG_subject_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Drop unwanted columns if present ---
df1 = df1.drop(columns=['contact_info', 'remarks'], errors='ignore')
df2 = df2.drop(columns=['contact_info', 'remarks'], errors='ignore')

# --- Drop rows with NaN in 'tstat' column ---
df1 = df1.dropna(subset=['tstat'])
df2 = df2.dropna(subset=['tstat'])

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Permutation Analysis Parameters ---
n_iter = 1000
perm_results = []

# --- Perform Permutation Test for Each Subject × Condition ---
for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    for condition in subj_data['cond'].unique():
        cond_data = subj_data[subj_data['cond'] == condition]
        session1 = cond_data[cond_data['session'] == 1]
        session2 = cond_data[cond_data['session'] == 2]

        merged = pd.merge(
            session1,
            session2,
            on=['source', 'detector'],
            suffixes=('_s1', '_s2')
        )

        if merged.shape[0] < 3:
            print(f"⚠️ Not enough matched channels for {subject}, {condition}. Skipping.")
            continue

        try:
            t2_s1 = merged['tstat_s1'] ** 2
            t2_s2 = merged['tstat_s2'] ** 2
            true_r, _ = pearsonr(t2_s1, t2_s2)

            perm_r_vals = []
            for _ in range(n_iter):
                s2_perm = np.random.permutation(t2_s2)
                r, _ = pearsonr(t2_s1, s2_perm)
                perm_r_vals.append(r)

            p_value = np.mean(np.array(perm_r_vals) >= true_r)

            perm_results.append({
                'subject': subject,
                'condition': condition,
                'true_r': true_r,
                'perm_r_vals': perm_r_vals,
                'p_value': p_value
            })
        except Exception as e:
            print(f"❌ Error for {subject}, {condition}: {e}")

# --- Create DataFrame for Results ---
perm_df = pd.DataFrame(perm_results)

# --- Plotting ---
plt.figure(figsize=(12, 6))
for _, row in perm_df.iterrows():
    sns.kdeplot(row['perm_r_vals'], fill=True, alpha=0.25, label=f"{row['subject']} ({row['condition']}) permuted")
    plt.axvline(row['true_r'], linestyle='--', linewidth=2, alpha=0.9)

plt.title("Permutation Test: Variance Stability (t² Correlation) by Subject & Condition")
plt.xlabel("Pearson r (Variance Stability)")
plt.ylabel("Density")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

# --- Display Results ---
print(perm_df[['subject', 'condition', 'true_r', 'p_value']])


✅ Found 25 subjects in both sessions.
   subject condition    true_r  p_value
0   pid001        x1 -0.249858    0.683
1   pid001        x2  0.041305    0.439
2   pid002        x1  0.152379    0.216
3   pid002        x2  0.139968    0.271
4   pid003        x1 -0.080672    0.606
5   pid003        x2  0.512887    0.095
6   pid004        x1 -0.158507    0.640
7   pid004        x2  0.282421    0.310
8   pid005        x1 -0.142060    0.661
9   pid005        x2 -0.030291    0.491
10  pid006        x1  0.682290    0.031
11  pid006        x2 -0.006754    0.528
12  pid007        x1  0.615433    0.109
13  pid007        x2  0.919592    0.002
14  pid008        x1  0.335321    0.227
15  pid008        x2  0.249741    0.271
16  pid009        x1 -0.779206    0.996
17  pid009        x2  0.586649    0.140
18  pid010        x1  0.195644    0.245
19  pid010        x2 -0.525884    0.999
20  pid011        x1  0.449056    0.142
21  pid011        x2  0.847748    0.020
22  pid012        x1 -0.228549    0.512
23

In [135]:
## Subject level spearman correlations 

import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2.csv"


df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Spearman r for Each Subject × Condition Across Channels ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    for condition in subj_data['cond'].unique():
        cond_data = subj_data[subj_data['cond'] == condition]

        session1 = cond_data[cond_data['session'] == 1]
        session2 = cond_data[cond_data['session'] == 2]

        merged = pd.merge(
            session1,
            session2,
            on=['source', 'detector'],  # Match on channel
            suffixes=('_s1', '_s2')
        )

        if merged.shape[0] < 3:
            print(f"⚠️ Not enough channels for {subject}, {condition}. Skipping.")
            continue

        try:
            r, p = spearmanr(merged['tstat_s1'], merged['tstat_s2'])
            participant_results.append({
                'subject': subject,
                'condition': condition,
                'spearman_r': r,
                'n_channels': merged.shape[0]
            })
        except Exception as e:
            print(f"❌ Error for {subject}, {condition}: {e}")

# --- Results DataFrame ---
subject_cond_corr_df = pd.DataFrame(participant_results)
print(subject_cond_corr_df)

# # --- Plot Histograms per Condition ---
# plt.figure(figsize=(8, 5))
# sns.histplot(data=subject_cond_corr_df, x='spearman_r', hue='condition', bins=10, kde=True, edgecolor='black')
# plt.title('Participant-Level Rank Test–Retest Correlations by Condition (Spearman)')
# plt.xlabel('Spearman r (Rank of T-values: Session 1 vs Session 2)')
# plt.ylabel('Number of Participants')
# plt.grid(True, linestyle='--', alpha=0.5)
# plt.tight_layout()
# plt.show()

# --- Individual Bar Plot: Spearman r per Subject × Condition ---
plt.figure(figsize=(10, 6))

# Sort subjects for consistent plotting
subject_order = sorted(subject_cond_corr_df['subject'].unique())

sns.barplot(
    data=subject_cond_corr_df,
    x='subject',
    y='spearman_r',
    hue='condition',
    order=subject_order,
    palette='Set2',
    edgecolor='black'
)

plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title('Session-to-Session Rank Correlation ', fontsize=14)
plt.ylabel('Spearman r')
plt.xlabel('Participant')
plt.xticks(rotation=45, ha='right')
plt.ylim(-.1, .8)
plt.legend(title='Condition')
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

✅ Found 26 subjects in both sessions.
   subject condition  spearman_r  n_channels
0   pid001        x1   -0.061486          46
1   pid001        x2    0.538082          46
2   pid002        x1    0.010299          46
3   pid002        x2    0.439901          46
4   pid003        x1    0.392044          46
5   pid003        x2    0.440765          46
6   pid004        x1   -0.013259          46
7   pid004        x2    0.475547          46
8   pid005        x1   -0.088005          46
9   pid005        x2    0.250817          46
10  pid006        x1    0.262165          46
11  pid006        x2    0.269935          46
12  pid007        x1    0.466297          46
13  pid007        x2    0.614801          46
14  pid008        x1    0.410546          46
15  pid008        x2    0.457169          46
16  pid009        x1    0.380327          46
17  pid009        x2    0.248844          46
18  pid010        x1    0.300401          46
19  pid010        x2    0.456429          46
20  pid011       

In [137]:
# --- Compute Group Average per Condition ---
group_avg = (
    subject_cond_corr_df
    .groupby('condition')['spearman_r']
    .mean()
    .reset_index()
    .rename(columns={'spearman_r': 'avg_spearman_r'})
)

print("📊 Average Spearman r by Condition:")
print(group_avg)

# --- Plot Group-Level Averages ---
plt.figure(figsize=(6, 5))
sns.barplot(
    data=group_avg,
    x='condition',
    y='avg_spearman_r',
    palette='pastel',
    edgecolor='black'
)

plt.title('Average Rank Correlation by Condition', fontsize=14)
plt.ylabel('Average Spearman r')
plt.xlabel('Condition')
plt.ylim(0, .6)  # adjust as needed
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()



📊 Average Spearman r by Condition:
  condition  avg_spearman_r
0        x1        0.284798
1        x2        0.336251


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\2450691587.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


In [139]:
## TASK VS REST correlation - pearson r 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r Across All Channels for Each Subject ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Match channels by source and detector
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_channels': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

# --- Plot Histogram of r Values ---
# --- Plot Barplot of Pearson r Values with Text Labels ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')

# Add text labels above each bar
for i, row in subject_corr_df.iterrows():
    ax.text(
        i,
        row['pearson_r'] + 0.02,  # offset for readability
        f"{row['pearson_r']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Test–Retest Correlations (All Channels)', fontsize=14)
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.2, max(subject_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Found 26 subjects in both sessions.
   subject  pearson_r  n_channels
0   pid001   0.511518          46
1   pid002   0.294712          46
2   pid003   0.540799          46
3   pid004   0.554181          46
4   pid005   0.432254          46
5   pid006   0.057141          46
6   pid007   0.593300          46
7   pid008   0.553703          46
8   pid009   0.347972          46
9   pid010   0.548505          46
10  pid011   0.548539          46
11  pid012   0.045052          46
12  pid013   0.258825          46
13  pid014   0.494320          46
14  pid015   0.651114          46
15  pid016   0.381356          46
16  pid017  -0.131296          46
17  pid018   0.256606          46
18  pid019   0.569127          46
19  pid020   0.384510          46
20  pid021   0.687765          46
21  pid022   0.414033          46
22  pid023   0.193038          46
23  pid024   0.594397          46
24  pid025   0.562858          46
25  pid026   0.540358          46


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\195188840.py:71: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')


In [141]:
print(np.mean(subject_corr_df['pearson_r']))

0.4186418014844225


In [145]:
## Subject Level Task vs Rest Correlation Permutation
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Permutation Parameters ---
n_iter = 1000
permutation_results = []

# --- Run Permutation Test Per Subject ---
for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Match channels by source and detector
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}. Skipping.")
        continue

    # True r
    s1_vec = merged['tstat_s1'].values
    s2_vec = merged['tstat_s2'].values
    true_r, _ = pearsonr(s1_vec, s2_vec)
    perm_r_vals = []

    for _ in range(n_iter):
        s2_perm = np.random.permutation(s2_vec)
        r, _ = pearsonr(s1_vec, s2_perm)
        perm_r_vals.append(r)

    permutation_results.append({
        'subject': subject,
        'true_r': true_r,
        'perm_r_vals': perm_r_vals,
        'p_value': np.mean(np.array(perm_r_vals) >= true_r)
    })

# --- Create DataFrame ---
perm_df = pd.DataFrame(permutation_results)

# --- Plotting ---
plt.figure(figsize=(12, 6))

for _, row in perm_df.iterrows():
    sns.kdeplot(row['perm_r_vals'], fill=True, alpha=0.2, label=f"{row['subject']} permuted")
    plt.axvline(row['true_r'], linestyle='--', linewidth=2, alpha=0.8, label=f"{row['subject']} true r")

plt.title("Within-Subject Permutation Distributions vs. True r Values Task vs Rest")
plt.xlabel("Pearson r")
plt.ylabel("Density")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# --- Print summary table ---
print(perm_df[['subject', 'true_r', 'p_value']])


✅ Found 26 subjects in both sessions.
   subject    true_r  p_value
0   pid001  0.511518    0.000
1   pid002  0.294712    0.028
2   pid003  0.540799    0.000
3   pid004  0.554181    0.000
4   pid005  0.432254    0.005
5   pid006  0.057141    0.350
6   pid007  0.593300    0.000
7   pid008  0.553703    0.000
8   pid009  0.347972    0.012
9   pid010  0.548505    0.000
10  pid011  0.548539    0.000
11  pid012  0.045052    0.371
12  pid013  0.258825    0.048
13  pid014  0.494320    0.000
14  pid015  0.651114    0.000
15  pid016  0.381356    0.006
16  pid017 -0.131296    0.806
17  pid018  0.256606    0.037
18  pid019  0.569127    0.000
19  pid020  0.384510    0.006
20  pid021  0.687765    0.000
21  pid022  0.414033    0.002
22  pid023  0.193038    0.090
23  pid024  0.594397    0.000
24  pid025  0.562858    0.000
25  pid026  0.540358    0.000


In [147]:
## Entire timeseries correlation - spearman rank 

import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_subject_glm_result_session2_tvr.csv"


df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Spearman r Across All Channels for Each Subject ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Match channels by source and detector
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}. Skipping.")
        continue

    try:
        r, p = spearmanr(merged['tstat_s1'], merged['tstat_s2'])
        participant_results.append({
            'subject': subject,
            'spearman_r': r,
            'n_channels': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

# --- Plot Barplot of Spearman r Values with Text Labels ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corr_df, x='subject', y='spearman_r', palette='viridis', edgecolor='black')

# Add text labels above each bar
for i, row in subject_corr_df.iterrows():
    ax.text(
        i,
        row['spearman_r'] + 0.02,  # slight offset above the bar
        f"{row['spearman_r']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Rank Correlations (All Channels, Spearman)', fontsize=14)
plt.xlabel('Subject')
plt.ylabel('Spearman r (Rank: Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.2, max(subject_corr_df['spearman_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()




✅ Found 26 subjects in both sessions.
   subject  spearman_r  n_channels
0   pid001    0.400925          46
1   pid002    0.249830          46
2   pid003    0.505273          46
3   pid004    0.409806          46
4   pid005    0.388467          46
5   pid006    0.089485          46
6   pid007    0.574468          46
7   pid008    0.605304          46
8   pid009    0.299784          46
9   pid010    0.651557          46
10  pid011    0.465187          46
11  pid012    0.004255          46
12  pid013    0.246747          46
13  pid014    0.510577          46
14  pid015    0.610854          46
15  pid016    0.345668          46
16  pid017   -0.083195          46
17  pid018    0.277212          46
18  pid019    0.543386          46
19  pid020    0.418686          46
20  pid021    0.614184          46
21  pid022    0.340487          46
22  pid023    0.187419          46
23  pid024    0.574591          46
24  pid025    0.525871          46
25  pid026    0.590626          46


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1014940819.py:71: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='spearman_r', palette='viridis', edgecolor='black')


In [149]:
print(np.mean(subject_corr_df['spearman_r']))

0.3979790792001708


In [44]:
## 1/2 of time series correlation - pearson r 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\HALF_glm_result_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\HALF_glm_result_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r Across All Channels for Each Subject ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Match channels by source and detector
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_channels': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

# --- Plot Histogram of r Values ---
# --- Plot Barplot of Pearson r Values with Text Labels ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')

# Add text labels above each bar
for i, row in subject_corr_df.iterrows():
    ax.text(
        i,
        row['pearson_r'] + 0.02,  # offset for readability
        f"{row['pearson_r']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Test–Retest Correlations (1/2 Time Series)', fontsize=14)
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.2, max(subject_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Found 11 subjects in both sessions.
    subject  pearson_r  n_channels
0   allison   0.522917          46
1      asha   0.199192          46
2     julia   0.074833          46
3    kareem   0.245418          46
4    lauren   0.362432          46
5   natalie   0.435498          46
6    nataly   0.269578          46
7      nick   0.275135          46
8        pk   0.175826          46
9        qi   0.109588          46
10    ricky   0.381261          46


C:\Users\pkell\AppData\Local\Temp\ipykernel_29240\1546460521.py:74: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')


In [46]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load the combined dataset ---
# Replace this with your actual path to the CSV file
file_path = r"C:\Users\pkell\Downloads\Combined_Pearson_r_Data.csv"
df = pd.read_csv(file_path)

# --- Sort subjects for consistent plotting ---
df['subject'] = df['subject'].astype(str)
subject_order = sorted(df['subject'].unique())
dataset_order = ['contrast', 'full', 'half']  # Ensure consistent order

# --- Plot grouped barplot ---
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=df,
    x='subject',
    y='pearson_r',
    hue='dataset',
    hue_order=dataset_order,
    order=subject_order,
    palette='Set2',
    edgecolor='black'
)

# Add text labels above each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", label_type="edge", padding=2, fontsize=8)

plt.title('Participant-Level Test–Retest Correlations Across Datasets', fontsize=14)
plt.xlabel('Participant')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.3, df['pearson_r'].max() + 0.1)
plt.legend(title='Dataset')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [577]:
## Functional Connectivity Within session correlation pearson r - TASK VS REST 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\73_odd_functional_conn_subjectlevel_ROI_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\73_even_functional_conn_subjectlevel_ROI_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Common Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r for Functional Connectivity Pattern for Each Subject ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Merge on the full connection pair (origin + destination)
    merged = pd.merge(
        session1,
        session2,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched pairs for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['R_s1'], merged['R_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_pairs': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

# --- Plot Pearson r Values for Each Subject ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')

# Add text labels above each bar
for i, row in subject_corr_df.iterrows():
    ax.text(
        i,
        row['pearson_r'] + 0.02,
        f"{row['pearson_r']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Within Session Correlation of Functional Connectivity', fontsize=14)
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, max(subject_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

✅ Found 73 subjects in both sessions.
                        subject  pearson_r  n_pairs
0                  allison2 rel   0.983120      100
1                     pid001rel   0.993425      100
2                     pid002rel   0.984004      100
3                     pid003rel   0.987444      100
4                     pid004rel   0.992410      100
5                     pid005rel   0.979493      100
6                     pid006rel   0.989145      100
7                     pid007rel   0.989646      100
8                     pid008rel   0.982737      100
9                     pid009rel   0.985105      100
10                    pid010rel   0.981037      100
11                    pid011rel   0.959021      100
12                    pid012rel   0.984559      100
13                    pid013rel   0.968578      100
14                    pid014rel   0.982537      100
15                    pid015rel   0.986167      100
16                    pid016rel   0.975514      100
17                    pid0

C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1091166487.py:71: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')


In [318]:
## Functional Connectivity session to session correlation pearson r - CONDITION SPLIT  
## Split by conditino and correlate session 1/2 condition 1/2
import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Common Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r separately for Condition 1 and 2 for Each Subject ---
participant_results = []

for subject in sorted(combined['subject'].unique()):
    subj_data = combined[combined['subject'] == subject]
    
    for condition in [1, 2]:  # Assuming conditions are labeled 1 and 2
        s1_cond = subj_data[(subj_data['session'] == 1) & (subj_data['condition'] == condition)]
        s2_cond = subj_data[(subj_data['session'] == 2) & (subj_data['condition'] == condition)]

        # Merge on the full connection pair (origin + destination)
        merged = pd.merge(
            s1_cond,
            s2_cond,
            on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
            suffixes=('_s1', '_s2')
        )

        if merged.shape[0] < 3:
            print(f"⚠️ Not enough matched pairs for {subject} condition {condition}. Skipping.")
            continue

        try:
            r, p = pearsonr(merged['R_s1'], merged['R_s2'])
            participant_results.append({
                'subject': subject,
                'condition': f'Condition {condition}',
                'pearson_r': r,
                'n_pairs': merged.shape[0]
            })
        except Exception as e:
            print(f"❌ Error for {subject} condition {condition}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

# --- Plot Pearson r Values for Each Subject × Condition ---
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 7))
ax = sns.barplot(
    data=subject_corr_df, 
    x='subject', 
    y='pearson_r', 
    hue='condition', 
    palette='Set2', 
    edgecolor='black'
)

# Add text labels above each bar
for container in ax.containers:
    ax.bar_label(
        container, 
        labels=[f"{h.get_height():.2f}" for h in container], 
        label_type="edge", 
        padding=2, 
        fontsize=8,
        rotation=0
    )

plt.title('Participant-Level Test–Retest Correlation of Functional Connectivity (Split by Condition)', fontsize=14)
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.3, max(subject_corr_df['pearson_r']) + 0.2)
plt.legend(title='Condition')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Found 26 subjects in both sessions.
   subject    condition  pearson_r  n_pairs
0   pid001  Condition 1   0.642080     2116
1   pid001  Condition 2   0.682012     2116
2   pid002  Condition 1   0.804988     2116
3   pid002  Condition 2   0.814995     2116
4   pid003  Condition 1   0.754747     2116
5   pid003  Condition 2   0.796699     2116
6   pid004  Condition 1   0.762095     2116
7   pid004  Condition 2   0.774365     2116
8   pid005  Condition 1   0.696813     2116
9   pid005  Condition 2   0.700653     2116
10  pid006  Condition 1   0.634168     2116
11  pid006  Condition 2   0.675282     2116
12  pid007  Condition 1   0.751348     2116
13  pid007  Condition 2   0.758027     2116
14  pid008  Condition 1   0.809106     2116
15  pid008  Condition 2   0.794359     2116
16  pid009  Condition 1   0.647463     2116
17  pid009  Condition 2   0.586479     2116
18  pid010  Condition 1   0.824391     2116
19  pid010  Condition 2   0.803925     2116
20  pid011  Condition 1   0.771908    

In [384]:
## Functional Connectivity session to session correlation pearson r - TASK VS REST 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Common Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r for Functional Connectivity Pattern for Each Subject ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Merge on the full connection pair (origin + destination)
    merged = pd.merge(
        session1,
        session2,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched pairs for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['R_s1'], merged['R_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_pairs': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

# --- Plot Pearson r Values for Each Subject ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')

# Add text labels above each bar
for i, row in subject_corr_df.iterrows():
    ax.text(
        i,
        row['pearson_r'] + 0.02,
        f"{row['pearson_r']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Test–Retest Correlation of Functional Connectivity', fontsize=14)
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, max(subject_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Found 26 subjects in both sessions.
   subject  pearson_r  n_pairs
0   pid001   0.667595     2116
1   pid002   0.820657     2116
2   pid003   0.787312     2116
3   pid004   0.787503     2116
4   pid005   0.709659     2116
5   pid006   0.671348     2116
6   pid007   0.772967     2116
7   pid008   0.815381     2116
8   pid009   0.636164     2116
9   pid010   0.822544     2116
10  pid011   0.778442     2116
11  pid012   0.554769     2116
12  pid013   0.714379     2116
13  pid014   0.854194     2116
14  pid015   0.796669     2116
15  pid016   0.538804     2116
16  pid017   0.952364     2116
17  pid018   0.176668     2116
18  pid019   0.817164     2116
19  pid020   0.815786     2116
20  pid021   0.885062     2116
21  pid022   0.773710     2116
22  pid023   0.768998     2116
23  pid024   0.745370     2116
24  pid025   0.651833     2116
25  pid026   0.906665     2116


C:\Users\pkell\AppData\Local\Temp\ipykernel_28364\1397176582.py:71: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')


In [168]:
print(np.mean(subject_corr_df['pearson_r']))

0.7393080123580881


In [322]:
## Functional Connectivity Task v Rest Permutation 
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_ROI_AGG_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_ROI_AGG_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Common Subjects ---
common_subjects = set(df1['subject']).intersection(df2['subject'])
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Permutation Parameters ---
n_iter = 1000
permutation_results = []

# --- Permutation Test Per Subject ---
for subject in sorted(combined['subject'].unique()):
    subj_data = combined[combined['subject'] == subject]
    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    merged = pd.merge(
        session1,
        session2,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Skipping {subject}: Not enough matched channel pairs.")
        continue

    s1_vec = merged['R_s1'].values
    s2_vec = merged['R_s2'].values

    try:
        true_r, _ = pearsonr(s1_vec, s2_vec)
        perm_r_vals = [pearsonr(s1_vec, np.random.permutation(s2_vec))[0] for _ in range(n_iter)]
        p_val = np.mean(np.array(perm_r_vals) >= true_r)

        permutation_results.append({
            'subject': subject,
            'true_r': true_r,
            'perm_r_vals': perm_r_vals,
            'p_value': p_val
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Create DataFrame ---
perm_df = pd.DataFrame(permutation_results)

# --- Clean perm_r_vals if any stored as string (prevent plotting errors) ---
def safe_parse(val):
    if isinstance(val, list):
        return val
    elif isinstance(val, str):
        try:
            return [float(x.strip()) for x in val.strip('[]').split(',')]
        except:
            return []
    return []

perm_df['perm_r_vals'] = perm_df['perm_r_vals'].apply(safe_parse)

# --- Drop rows with invalid permutation vectors ---
perm_df = perm_df[perm_df['perm_r_vals'].apply(lambda x: len(x) > 5)].reset_index(drop=True)

# --- Plot ---
plt.figure(figsize=(14, 7))
for _, row in perm_df.iterrows():
    sns.kdeplot(row['perm_r_vals'], fill=True, alpha=0.2, label=f"{row['subject']} permuted")
    plt.axvline(row['true_r'], linestyle='--', linewidth=2, alpha=0.9, label=f"{row['subject']} true r")

plt.title("Within-Subject Permutation Distributions vs. True r Values (Functional Connectivity)", fontsize=14)
plt.xlabel("Pearson r")
plt.ylabel("Density")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

# --- Output Summary Table ---
print(perm_df[['subject', 'true_r', 'p_value']])


   subject    true_r  p_value
0   pid001  0.843653      0.0
1   pid002  0.913117      0.0
2   pid003  0.916637      0.0
3   pid004  0.822661      0.0
4   pid005  0.910607      0.0
5   pid006  0.934384      0.0
6   pid007  0.957217      0.0
7   pid008  0.953805      0.0
8   pid009  0.936296      0.0
9   pid010  0.942815      0.0
10  pid011  0.932412      0.0
11  pid012  0.884262      0.0
12  pid013  0.874480      0.0
13  pid014  0.960001      0.0
14  pid015  0.946386      0.0
15  pid016  0.671583      0.0
16  pid017  0.980907      0.0
17  pid018  0.490508      0.0
18  pid019  0.971938      0.0
19  pid020  0.940483      0.0
20  pid021  0.957959      0.0
21  pid022  0.948413      0.0
22  pid023  0.923248      0.0
23  pid024  0.968595      0.0
24    theo  0.979158      0.0


In [320]:
## Functional Connectivity Condition Split Permutation 

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_ROI_AGG_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_ROI_AGG_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Common Subjects ---
common_subjects = set(df1['subject']).intersection(df2['subject'])
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Permutation Parameters ---
n_iter = 1000
permutation_results = []

# --- Permutation Test Per Subject and Condition ---
for subject in sorted(combined['subject'].unique()):
    subj_data = combined[combined['subject'] == subject]
    
    for cond in subj_data['condition'].unique():
        cond_data = subj_data[subj_data['condition'] == cond]
        session1 = cond_data[cond_data['session'] == 1]
        session2 = cond_data[cond_data['session'] == 2]

        merged = pd.merge(
            session1,
            session2,
            on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
            suffixes=('_s1', '_s2')
        )

        if merged.shape[0] < 3:
            print(f"⚠️ Skipping {subject}, condition {cond}: Not enough matched pairs.")
            continue

        s1_vec = merged['R_s1'].values
        s2_vec = merged['R_s2'].values

        try:
            true_r, _ = pearsonr(s1_vec, s2_vec)
            perm_r_vals = [pearsonr(s1_vec, np.random.permutation(s2_vec))[0] for _ in range(n_iter)]
            p_val = np.mean(np.array(perm_r_vals) >= true_r)

            permutation_results.append({
                'subject': subject,
                'condition': cond,
                'true_r': true_r,
                'perm_r_vals': perm_r_vals,
                'p_value': p_val
            })
        except Exception as e:
            print(f"❌ Error for {subject}, condition {cond}: {e}")

# --- Create DataFrame ---
perm_df = pd.DataFrame(permutation_results)

# --- Clean perm_r_vals if any stored as string (prevent plotting errors) ---
def safe_parse(val):
    if isinstance(val, list):
        return val
    elif isinstance(val, str):
        try:
            return [float(x.strip()) for x in val.strip('[]').split(',')]
        except:
            return []
    return []

perm_df['perm_r_vals'] = perm_df['perm_r_vals'].apply(safe_parse)
perm_df = perm_df[perm_df['perm_r_vals'].apply(lambda x: len(x) > 5)].reset_index(drop=True)

# --- Plot KDEs with True r Values, Separated by Condition ---
g = sns.FacetGrid(perm_df, col="condition", col_wrap=2, height=4, sharex=True, sharey=True)

def plot_kde(data, **kwargs):
    for _, row in data.iterrows():
        sns.kdeplot(row['perm_r_vals'], fill=True, alpha=0.2, label=f"{row['subject']} permuted", **kwargs)
        plt.axvline(row['true_r'], linestyle='--', linewidth=2, alpha=0.9, label=f"{row['subject']} true r")
    plt.xlabel("Pearson r")
    plt.ylabel("Density")
    plt.grid(True, linestyle='--', alpha=0.4)

g.map_dataframe(plot_kde)
g.set_titles(col_template="Condition: {col_name}")
g.add_legend(fontsize=7, bbox_to_anchor=(1.05, 0.5), loc='center left')
plt.tight_layout()
plt.show()

# --- Output Summary Table ---
print("\n📊 Permutation-Based FC Test–Retest Reliability:")
print(perm_df[['subject', 'condition', 'true_r', 'p_value']])



📊 Permutation-Based FC Test–Retest Reliability:
   subject  condition    true_r  p_value
0   pid001          1  0.800690      0.0
1   pid001          2  0.867218      0.0
2   pid002          1  0.917529      0.0
3   pid002          2  0.894351      0.0
4   pid003          1  0.882965      0.0
5   pid003          2  0.924855      0.0
6   pid004          1  0.833528      0.0
7   pid004          2  0.786665      0.0
8   pid005          1  0.916641      0.0
9   pid005          2  0.887989      0.0
10  pid006          1  0.909687      0.0
11  pid006          2  0.932546      0.0
12  pid007          1  0.956403      0.0
13  pid007          2  0.932608      0.0
14  pid008          1  0.945742      0.0
15  pid008          2  0.937871      0.0
16  pid009          1  0.930458      0.0
17  pid009          2  0.921672      0.0
18  pid010          1  0.923518      0.0
19  pid010          2  0.935457      0.0
20  pid011          1  0.923940      0.0
21  pid011          2  0.923626      0.0
22  pid0

✅ Found 26 subjects in both sessions.
   subject    condition  pearson_r  n_pairs
0   pid001  Condition 1   0.642080     2116
1   pid001  Condition 2   0.682012     2116
2   pid002  Condition 1   0.804988     2116
3   pid002  Condition 2   0.814995     2116
4   pid003  Condition 1   0.754747     2116
5   pid003  Condition 2   0.796699     2116
6   pid004  Condition 1   0.762095     2116
7   pid004  Condition 2   0.774365     2116
8   pid005  Condition 1   0.696813     2116
9   pid005  Condition 2   0.700653     2116
10  pid006  Condition 1   0.634168     2116
11  pid006  Condition 2   0.675282     2116
12  pid007  Condition 1   0.751348     2116
13  pid007  Condition 2   0.758027     2116
14  pid008  Condition 1   0.809106     2116
15  pid008  Condition 2   0.794359     2116
16  pid009  Condition 1   0.647463     2116
17  pid009  Condition 2   0.586479     2116
18  pid010  Condition 1   0.824391     2116
19  pid010  Condition 2   0.803925     2116
20  pid011  Condition 1   0.771908    

In [199]:
## ROI Functional Connectivity session to session correlation pearson r 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# Define Regions of Interest (ROI)
roi_definitions = {
    "Left RLPFC": [
        'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'
    ],
    "Left DLPFC": [
          'S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo','S5_D3 hbo', 'S5_D4 hbo', 'S5_D16 hbo','S2_D3 hbo'
    ],
    "Left VLPFC": [
         'S3_D1 hbo', 'S3_D3 hbo'
    ],
    "Left Sup Parietal": [
         'S12_D8 hbo', 'S12_D10 hbo', 'S13_D10 hbo','S13_D20 hbo', 'S12_D19 hbo'
    ],
    "Left Inf Parietal": [
        'S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo', 'S13_D9 hbo', 'S13_D11 hbo', 'S11_D18 hbo',
    ],
    "Right RLPFC": [
        'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo', 
    ],
    "Right DLPFC": [
         'S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo', 'S10_D7 hbo', 'S10_D17 hbo'
    ],
    "Right VLPFC": [
         'S9_D7 hbo', 'S9_D6 hbo'
    ],
    "Right Sup Parietal": [
        'S14_D12 hbo', 'S14_D13 hbo', 
         'S14_D21 hbo', 'S16_D13 hbo'
    ],
    "Right Inf Parietal": [
         'S15_D12 hbo', 'S15_D14 hbo','S16_D12 hbo', 'S16_D14 hbo', 'S16_D15 hbo', 'S15_D22 hbo','S16_D23 hbo' 
    ]
}

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_functional_conn_subjectlevel_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Adjusted column names to match the actual data structure
df1['channel'] = 'S' + df1['SourceOrigin'].astype(str) + '_D' + df1['DetectorOrigin'].astype(str) + ' hbo'
df2['channel'] = 'S' + df2['SourceOrigin'].astype(str) + '_D' + df2['DetectorOrigin'].astype(str) + ' hbo'

# Filter to only matching subjects again just in case
common_subjects = set(df1['subject']).intersection(df2['subject'])
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# Prepare a DataFrame to store ROI-level test–retest correlations
roi_corr_results = []

# Iterate over subjects
for subject in sorted(common_subjects):
    df1_sub = df1[df1['subject'] == subject]
    df2_sub = df2[df2['subject'] == subject]

    for roi, channels in roi_definitions.items():
        vec1 = df1_sub[df1_sub['channel'].isin(channels)].sort_values('channel')['R'].values
        vec2 = df2_sub[df2_sub['channel'].isin(channels)].sort_values('channel')['R'].values

        # Ensure equal length before correlation
        if len(vec1) > 2 and len(vec1) == len(vec2):
            r_val = pd.Series(vec1).corr(pd.Series(vec2))
            roi_corr_results.append({
                'subject': subject,
                'roi': roi,
                'pearson_r': r_val,
                'n_channels': len(vec1)
            })

roi_corr_df = pd.DataFrame(roi_corr_results)

In [201]:
# Recreate the previous ROI-level correlation file for plotting and statistical comparison
import pandas as pd

# Load the original subject-level ROI correlation data
df = roi_corr_df

# Pivot the data for statistical testing
df_wide = df.pivot(index='subject', columns='roi', values='pearson_r')

# --- Statistical Analysis ---
from scipy.stats import f_oneway, ttest_rel

# Repeated-measures ANOVA across the 4 ROIs
anova_result = f_oneway(
    df_wide['Left Frontal'],
    df_wide['Right Frontal'],
    df_wide['Left Parietal'],
    df_wide['Right Parietal']
)

# Pairwise t-tests between ROIs
pairwise_results = []
rois = df['roi'].unique()
for i in range(len(rois)):
    for j in range(i + 1, len(rois)):
        roi1 = rois[i]
        roi2 = rois[j]
        t_stat, p_val = ttest_rel(df_wide[roi1], df_wide[roi2])
        pairwise_results.append({
            'ROI_1': roi1,
            'ROI_2': roi2,
            't_stat': t_stat,
            'p_value': p_val
        })

pairwise_df = pd.DataFrame(pairwise_results)

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.barplot(data=df, x='subject', y='pearson_r', hue='roi', edgecolor='black')
plt.title('ROI-Level Test–Retest Functional Connectivity Correlations')
plt.xlabel('Participant')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.2, 1.0)
plt.legend(title='ROI', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

# Return stats
anova_result.statistic, anova_result.pvalue, pairwise_df


KeyError: 'Left Frontal'

In [202]:
## Group Level Functional Connectivity Correlations

# Define Regions of Interest (ROI)
roi_definitions = {
    "Left RLPFC": [
        'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'
    ],
    "Left DLPFC": [
          'S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo','S5_D3 hbo', 'S5_D4 hbo', 'S5_D16 hbo','S2_D3 hbo'
    ],
    "Left VLPFC": [
         'S3_D1 hbo', 'S3_D3 hbo'
    ],
    "Left Sup Parietal": [
         'S12_D8 hbo', 'S12_D10 hbo', 'S13_D10 hbo','S13_D20 hbo', 'S12_D19 hbo'
    ],
    "Left Inf Parietal": [
        'S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo', 'S13_D9 hbo', 'S13_D11 hbo', 'S11_D18 hbo',
    ],
    "Right RLPFC": [
        'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo', 
    ],
    "Right DLPFC": [
         'S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo', 'S10_D7 hbo', 'S10_D17 hbo'
    ],
    "Right VLPFC": [
         'S9_D7 hbo', 'S9_D6 hbo'
    ],
    "Right Sup Parietal": [
        'S14_D12 hbo', 'S14_D13 hbo', 
         'S14_D21 hbo', 'S16_D13 hbo'
    ],
    "Right Inf Parietal": [
         'S15_D12 hbo', 'S15_D14 hbo','S16_D12 hbo', 'S16_D14 hbo', 'S16_D15 hbo', 'S15_D22 hbo','S16_D23 hbo' 
    ]
}


# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session2_tvr.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Construct channel name
df1['channel'] = 'S' + df1['SourceOrigin'].astype(str) + '_D' + df1['DetectorOrigin'].astype(str) + ' hbo'
df2['channel'] = 'S' + df2['SourceOrigin'].astype(str) + '_D' + df2['DetectorOrigin'].astype(str) + ' hbo'

# Merge on channel, ROI, TypeOrigin/Dest to align connectivity pairs
merge_cols = ['channel', 'SourceDest', 'DetectorDest', 'TypeOrigin', 'TypeDest']
merged = pd.merge(
    df1, df2,
    on=merge_cols,
    suffixes=('_s1', '_s2')
)

# Drop any rows with missing r values
merged = merged.dropna(subset=['R_s1', 'R_s2'])

# Compute the group-level test–retest correlation
group_r, group_p = pearsonr(merged['R_s1'], merged['R_s2'])

# Prepare final result
group_result_df = pd.DataFrame([{
    'pearson_r': group_r,
    'p_value': group_p,
    'n_channel_pairs': merged.shape[0]
}])

In [204]:
group_result_df

,pearson_r,p_value,n_channel_pairs
0,0.978749,0.0,8372


In [191]:
import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# # --- ROI Definitions ---
# roi_definitions = {
#     "Left Frontal": [
#         'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo', 'S2_D3 hbo',
#         'S3_D1 hbo', 'S3_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo',
#         'S5_D3 hbo', 'S5_D4 hbo', 'S5_D16 hbo'
#     ],
#     "Left Parietal": [
#         'S11_D9 hbo', 'S11_D8 hbo', 'S12_D8 hbo', 'S12_D10 hbo', 'S13_D8 hbo',
#         'S13_D9 hbo', 'S13_D10 hbo', 'S13_D11 hbo', 'S12_D19 hbo', 'S11_D18 hbo', 'S13_D20 hbo'
#     ],
#     "Right Frontal": [
#         'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo', 'S7_D7 hbo',
#         'S8_D5 hbo', 'S8_D7 hbo', 'S9_D7 hbo', 'S9_D6 hbo', 'S10_D7 hbo', 'S10_D17 hbo'
#     ],
#     "Right Parietal": [
#         'S14_D12 hbo', 'S14_D13 hbo', 'S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo',
#         'S16_D13 hbo', 'S16_D14 hbo', 'S16_D15 hbo', 'S14_D21 hbo', 'S15_D22 hbo', 'S16_D23 hbo'
#     ]
# }
roi_definitions = {
    "Left RLPFC": [
        'S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'
    ],
    "Left DLPFC": [
          'S4_D3 hbo', 'S4_D2 hbo', 'S4_D4 hbo','S5_D3 hbo', 'S5_D4 hbo', 'S5_D16 hbo','S2_D3 hbo'
    ],
    "Left VLPFC": [
         'S3_D1 hbo', 'S3_D3 hbo'
    ],
    "Left Sup Parietal": [
         'S12_D8 hbo', 'S12_D10 hbo', 'S13_D10 hbo','S13_D20 hbo', 'S12_D19 hbo'
    ],
    "Left Inf Parietal": [
        'S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo', 'S13_D9 hbo', 'S13_D11 hbo', 'S11_D18 hbo',
    ],
    "Right RLPFC": [
        'S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo', 
    ],
    "Right DLPFC": [
         'S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo', 'S10_D7 hbo', 'S10_D17 hbo'
    ],
    "Right VLPFC": [
         'S9_D7 hbo', 'S9_D6 hbo'
    ],
    "Right Sup Parietal": [
        'S14_D12 hbo', 'S14_D13 hbo', 
         'S14_D21 hbo', 'S16_D13 hbo'
    ],
    "Right Inf Parietal": [
         'S15_D12 hbo', 'S15_D14 hbo','S16_D12 hbo', 'S16_D14 hbo', 'S16_D15 hbo', 'S15_D22 hbo','S16_D23 hbo' 
    ]
}
# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session1_tvr.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\FINAL_group_functional_conn_session2_tvr.csv"


df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Construct channel name ---
df1['channel'] = 'S' + df1['SourceOrigin'].astype(str) + '_D' + df1['DetectorOrigin'].astype(str) + ' hbo'
df2['channel'] = 'S' + df2['SourceOrigin'].astype(str) + '_D' + df2['DetectorOrigin'].astype(str) + ' hbo'

# --- Filter matching channel pairs across sessions ---
merge_cols = ['channel', 'SourceDest', 'DetectorDest', 'TypeOrigin', 'TypeDest']
merged = pd.merge(df1, df2, on=merge_cols, suffixes=('_s1', '_s2'))
merged = merged.dropna(subset=['R_s1', 'R_s2'])

# --- Map channel to ROI label ---
roi_map = {}
for roi, chans in roi_definitions.items():
    for ch in chans:
        roi_map[ch] = roi

# Filter to channels in ROI map
merged = merged[merged['channel'].isin(roi_map)]

# Assign ROI
merged['ROI'] = merged['channel'].map(roi_map)

# --- Aggregate R values per ROI for each session ---
roi_s1 = merged.groupby('ROI')['R_s1'].mean()
roi_s2 = merged.groupby('ROI')['R_s2'].mean()

# --- Correlate ROI-level vectors across sessions ---
group_roi_r, group_roi_p = pearsonr(roi_s1, roi_s2)

# --- Results ---
print("Group-level ROI Test–Retest Correlation")
print(f"Pearson r: {group_roi_r:.3f}, p-value: {group_roi_p:.4f}")

# --- Plot ROI-level correlation with stats annotation ---
plt.figure(figsize=(6, 6))
sns.regplot(x=roi_s1, y=roi_s2)

# Annotate ROI labels
for label in roi_s1.index:
    plt.text(roi_s1[label], roi_s2[label], label, fontsize=9, ha='right', va='bottom')

# Annotate Pearson r and p-value
plt.annotate(
    f"r = {group_roi_r:.2f}\np = {group_roi_p:.4f}",
    xy=(0.05, 0.95),
    xycoords='axes fraction',
    fontsize=11,
    ha='left',
    va='top',
    bbox=dict(boxstyle="round,pad=0.3", edgecolor='gray', facecolor='white', alpha=0.8)
)

plt.title('Group ROI-Level FC: Session 1 vs Session 2', fontsize=13)
plt.xlabel('Mean R (Session 1)')
plt.ylabel('Mean R (Session 2)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


Group-level ROI Test–Retest Correlation
Pearson r: 0.991, p-value: 0.0000


In [67]:
## Contrast Correlations

## Participant Session to Session Correlations 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# --- Load Data ---
file1 = r"C:\Users\pkell\Downloads\updated_subjectIDs_session1.csv"
file2 = r"C:\Users\pkell\Downloads\updated_subjectIDs_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['Chromophore'].str.lower() == 'hbo'].copy()
df2 = df2[df2['Chromophore'].str.lower() == 'hbo'].copy()


# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['SubjectID']).intersection(set(df2['SubjectID']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['SubjectID'].isin(common_subjects)]
df2 = df2[df2['SubjectID'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r for Each Participant Across Channels ---
participant_results = []

for subject in combined['SubjectID'].unique():
    subj_data = combined[combined['SubjectID'] == subject]
    
    session1 = subj_data[subj_data['session'] == 1].copy()
    session2 = subj_data[subj_data['session'] == 2].copy()

    # Match channels by source and detector
    merged = pd.merge(
        session1,
        session2,
        on=['Channel'],  # Match on channel and condition
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough channels to compute correlation for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['T_s1'], merged['T_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_channels': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

# --- Plot Histogram of r Values ---
plt.figure(figsize=(8, 5))
plt.hist(subject_corr_df['pearson_r'], bins=10, edgecolor='black')
plt.title('Participant-Level Contrast Test–Retest Correlations')
plt.xlabel('Pearson r (Session 1 vs Session 2)')
plt.ylabel('Number of Participants')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

✅ Found 9 subjects in both sessions.
   subject  pearson_r  n_channels
0  allison   0.081774          46
1     asha  -0.381884          46
2    julia  -0.105995          46
3  natalie  -0.226951          46
4   nataly   0.359472          46
5     nick   0.191103          46
6       pk  -0.251536          46
7       qi   0.264150          46
8    ricky   0.326379          46


In [69]:
# --- Plot Bar Plot of Individual r Values ---
plt.figure(figsize=(10, 6))
sorted_df = subject_corr_df.sort_values(by='pearson_r', ascending=False)

bars = plt.bar(x=sorted_df['subject'], height=sorted_df['pearson_r'], color='skyblue', edgecolor='black')
plt.axhline(0, color='gray', linestyle='--', linewidth=1)

# Add r value labels above bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=8)

plt.title('Participant-Level Contrast Test–Retest Correlations')
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-1, 1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [43]:
con_mean = np.mean(subject_corr_df['pearson_r'])

In [45]:
con_mean

0.07979952761066508

In [142]:
file = r"C:\Users\pkell\OneDrive\Documents\MATLAB\glm_resultsallison1.csv"
df1 = pd.read_csv(file)

print(df1.head())


   source  detector type  ShortSeperation cond      beta        se     tstat  \
0       1         1  hbo                0   x1  1.739655  2.253856  0.771857   
1       1         1  hbr                0   x1  0.138040  0.715456  0.192940   
2       1         2  hbo                0   x1  1.444402  1.688462  0.855454   
3       1         2  hbr                0   x1 -1.316768  0.587488 -2.241354   
4       2         1  hbo                0   x1  4.625987  2.248757  2.057131   

           dfe         p         q  minDiscoverableChange  RelativePower  
0  1380.606483  0.440331  0.729918               5.607236       0.090044  
1  1380.606483  0.847035  0.916791               1.779942       0.283659  
2  1380.606483  0.392448  0.697230               4.200627       0.120195  
3  1380.606483  0.025162  0.257207               1.461576       0.345446  
4  1380.606483  0.039861  0.318889               5.594551       0.090248  


In [124]:
# file = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3 data\glm_resultsa1.csv"
# df1 = pd.read_csv(file)

# print(df1.head())

file = r"C:\Users\pkell\OneDrive\Documents\MATLAB\glm_resultsp1.csv"
df1 = pd.read_csv(file)

print(df1.head())

   source  detector type  ShortSeperation cond      beta        se     tstat  \
0       1         1  hbo                0   x1 -7.149034  5.982561 -1.194979   
1       1         1  hbr                0   x1  0.404174  1.033006  0.391260   
2       1         2  hbo                0   x1 -6.133637  4.210103 -1.456885   
3       1         2  hbr                0   x1  1.129683  1.038847  1.087439   
4       2         1  hbo                0   x1 -1.432044  6.807661 -0.210358   

           dfe         p         q  minDiscoverableChange  RelativePower  
0  1713.308067  0.232261  0.931035              14.882075       0.078968  
1  1713.308067  0.695653  0.964489               2.569681       0.457337  
2  1713.308067  0.145331  0.931035              10.472950       0.112214  
3  1713.308067  0.276996  0.964489               2.584210       0.454766  
4  1713.308067  0.833413  0.970557              16.934573       0.069397  


NameError: name 'std' is not defined

In [144]:
file = r"C:\Users\pkell\OneDrive\Documents\MATLAB\glm_resultsallison2.csv"
df2 = pd.read_csv(file)

print(df2.head())

# file = r"C:\Users\pkell\OneDrive\Documents\MATLAB\glm_resultsa2.csv"
# df2 = pd.read_csv(file)

# print(df2.head())


   source  detector type  ShortSeperation cond      beta        se     tstat  \
0       1         1  hbo                0   x1 -1.264106  2.328805 -0.542813   
1       1         1  hbr                0   x1  0.496923  0.656414  0.757026   
2       1         2  hbo                0   x1  0.491196  2.004239  0.245079   
3       1         2  hbr                0   x1  0.590779  0.627527  0.941440   
4       2         1  hbo                0   x1  1.598103  2.699686  0.591959   

           dfe         p         q  minDiscoverableChange  RelativePower  
0  1369.923286  0.587347  0.878879               5.793722       0.149587  
1  1369.923286  0.449164  0.825447               1.633062       0.530701  
2  1369.923286  0.806432  0.916455               4.986250       0.173812  
3  1369.923286  0.346645  0.807377               1.561195       0.555131  
4  1369.923286  0.553976  0.863827               6.716421       0.129037  


In [164]:
df2x2 = df2[df2['type'] == 'hbo']
df2x2 = df2[df2['cond'] == 'x2']
df1x2 = df1[df1['type'] == 'hbo']
df1x2 = df1[df1['cond'] == 'x2']
# df2 = df2[df2['ShortSeperation'] == 0]

In [166]:
df2x1 = df2[df2['type'] == 'hbo']
df2x1 = df2[df2['cond'] == 'x1']
df1x1 = df1[df1['type'] == 'hbo']
df1x1 = df1[df1['cond'] == 'x1']

In [168]:
merged_dfx1 = pd.merge(
    df1x1, df2x1,
    left_index=True, right_index=True,
    suffixes=('_Session1', '_Session2')
)
merged_dfx1

,source_Session1,detector_Session1,type_Session1,ShortSeperation_Session1,cond_Session1,beta_Session1,se_Session1,tstat_Session1,dfe_Session1,p_Session1,...,ShortSeperation_Session2,cond_Session2,beta_Session2,se_Session2,tstat_Session2,dfe_Session2,p_Session2,q_Session2,minDiscoverableChange_Session2,RelativePower_Session2
0,1,1,hbo,0,x1,1.739655,2.253856,0.771857,1380.606483,0.440331,...,0,x1,-1.264106,2.328805,-0.542813,1369.923286,0.587347,0.878879,5.793722,0.149587
2,1,2,hbo,0,x1,1.444402,1.688462,0.855454,1380.606483,0.392448,...,0,x1,0.491196,2.004239,0.245079,1369.923286,0.806432,0.916455,4.986250,0.173812
4,2,1,hbo,0,x1,4.625987,2.248757,2.057131,1380.606483,0.039861,...,0,x1,1.598103,2.699686,0.591959,1369.923286,0.553976,0.863827,6.716421,0.129037
6,2,2,hbo,0,x1,4.305026,2.790558,1.542712,1380.606483,0.123130,...,0,x1,2.317841,4.153182,0.558088,1369.923286,0.576876,0.877232,10.332504,0.083878
8,2,3,hbo,0,x1,1.457936,0.790328,1.844722,1380.606483,0.065292,...,0,x1,1.104572,1.411777,0.782399,1369.923286,0.434116,0.825447,3.512292,0.246753
10,3,1,hbo,0,x1,1.215310,1.763587,0.689112,1380.606483,0.490868,...,0,x1,-2.477360,2.555878,-0.969280,1369.923286,0.332577,0.806676,6.358646,0.136298
12,3,3,hbo,0,x1,0.258845,1.211669,0.213627,1380.606483,0.830869,...,0,x1,-1.760892,2.009564,-0.876256,1369.923286,0.381045,0.825447,4.999499,0.173351
14,4,2,hbo,0,x1,0.961334,2.033613,0.472722,1380.606483,0.636486,...,0,x1,4.098955,2.152267,1.904482,1369.923286,0.057057,0.666014,5.354523,0.161857
16,4,3,hbo,0,x1,1.438804,1.327607,1.083758,1380.606483,0.278662,...,0,x1,2.861035,2.239788,1.277368,1369.923286,0.201689,0.742214,5.572263,0.155533
18,4,4,hbo,0,x1,0.183279,1.835404,0.099857,1380.606483,0.920472,...,0,x1,0.696026,2.399347,0.290090,1369.923286,0.771792,0.912825,5.969222,0.145189


In [170]:
merged_dfx2 = pd.merge(
    df1x2, df2x2,
    left_index=True, right_index=True,
    suffixes=('_Session1', '_Session2')
)
merged_dfx2

,source_Session1,detector_Session1,type_Session1,ShortSeperation_Session1,cond_Session1,beta_Session1,se_Session1,tstat_Session1,dfe_Session1,p_Session1,...,ShortSeperation_Session2,cond_Session2,beta_Session2,se_Session2,tstat_Session2,dfe_Session2,p_Session2,q_Session2,minDiscoverableChange_Session2,RelativePower_Session2
92,1,1,hbo,0,x2,2.245295,2.257068,0.994784,1380.606483,0.320016,...,0,x2,1.740159,2.291491,0.759400,1369.923286,0.447744,0.825447,5.700893,0.152023
94,1,2,hbo,0,x2,2.227568,1.700927,1.309620,1380.606483,0.190542,...,0,x2,1.316455,1.984419,0.663396,1369.923286,0.507189,0.825447,4.936942,0.175548
96,2,1,hbo,0,x2,4.231586,2.225728,1.901214,1380.606483,0.057482,...,0,x2,3.074188,2.660167,1.155637,1369.923286,0.248031,0.779913,6.618102,0.130954
98,2,2,hbo,0,x2,4.546267,2.759051,1.647765,1380.606483,0.099628,...,0,x2,4.867199,4.082816,1.192118,1369.923286,0.233421,0.779913,10.157445,0.085323
100,2,3,hbo,0,x2,2.048559,0.788684,2.597441,1380.606483,0.009492,...,0,x2,1.280544,1.391553,0.920226,1369.923286,0.357617,0.811738,3.461978,0.250339
102,3,1,hbo,0,x2,1.538469,1.723682,0.892548,1380.606483,0.372255,...,0,x2,0.982499,2.535369,0.387517,1369.923286,0.698434,0.891959,6.307625,0.137400
104,3,3,hbo,0,x2,1.406498,1.199071,1.172990,1380.606483,0.241002,...,0,x2,2.452598,1.953199,1.255683,1369.923286,0.209445,0.755645,4.859270,0.178354
106,4,2,hbo,0,x2,3.210995,2.022066,1.587978,1380.606483,0.112520,...,0,x2,0.804292,2.145021,0.374958,1369.923286,0.707750,0.891959,5.336496,0.162404
108,4,3,hbo,0,x2,3.144703,1.309033,2.402310,1380.606483,0.016423,...,0,x2,2.166459,2.228173,0.972303,1369.923286,0.331071,0.806676,5.543364,0.156343
110,4,4,hbo,0,x2,3.257321,1.827626,1.782269,1380.606483,0.074925,...,0,x2,-0.729582,2.371073,-0.307701,1369.923286,0.758356,0.912010,5.898879,0.146921


In [172]:
# Extract beta values for correlation
beta_session1x1 = merged_dfx1['beta_Session1']
beta_session2x1 = merged_dfx1['beta_Session2']

# Compute Pearson correlation
corr, p_value = stat.pearsonr(beta_session1, beta_session2)
print(f"Correlation: {corr}, p-value: {p_value}")


Correlation: 0.14645175283469566, p-value: 0.1636097639912549


In [174]:
# Extract beta values for correlation
beta_session1x2 = merged_dfx2['beta_Session1']
beta_session2x2 = merged_dfx2['beta_Session2']

# Compute Pearson correlation
corr, p_value = stat.pearsonr(beta_session1, beta_session2)
print(f"Correlation: {corr}, p-value: {p_value}")


Correlation: 0.14645175283469566, p-value: 0.1636097639912549


In [113]:
from sklearn.preprocessing import StandardScaler

# Extract beta values for correlation
beta_session1 = merged_df['beta_Session1']
beta_session2 = merged_df['beta_Session2']

scaler = StandardScaler()
beta_session1_scaled = scaler.fit_transform(beta_session1.values.reshape(-1, 1)).flatten()
beta_session2_scaled = scaler.fit_transform(beta_session2.values.reshape(-1, 1)).flatten()


# Compute Pearson correlation
corr, p_value = stat.pearsonr(beta_session1_scaled, beta_session2_scaled)
print(f"Correlation: {corr}, p-value: {p_value}")


Correlation: 0.28952975134410386, p-value: 0.050977227733354336


In [154]:
# Extract beta values for correlation
t1 = merged_df['tstat_Session1']
t2 = merged_df['tstat_Session2']

# Compute Pearson correlation
corr, p_value = stat.pearsonr(t1, t2)
print(f"Correlation: {corr}, p-value: {p_value}")


Correlation: 0.12718386426187556, p-value: 0.22699393123573638


In [162]:
### Beta values are much higher for session 1 need to figure out why that is??????

### try to run whole MNE analysis then load in df from MATLAB and manually change the column values in MNE df to values in Matlab DF to plot



import matplotlib.pyplot as plt

plt.scatter(beta_session1, beta_session2, color='blue', label=f'beta values')
plt.xlabel('Session 1 Beta Values')
plt.ylabel('Session 2 Beta Values')
plt.title('Channel-Wise Beta Value Correlation Between Sessions')
plt.axline((0, 0), slope= corr, color='red', linestyle='--', label=f'r = {corr:.2f}, p = {p_value: .3f} ')
plt.legend()
plt.show()


In [75]:
subject_file_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\snirf\pid004.snirf"


raw_intensity = mne.io.read_raw_snirf(subject_file_path, verbose=False)

# raw_intensity.annotations.set_durations(24)## kee if they are the same length
for i, desc in enumerate(raw_intensity.annotations.description):
    if desc in ['1','2',]:
        raw_intensity.annotations.duration[i] = 36
        
# # Print the updated annotations to verify
unwanted = np.nonzero((raw_intensity.annotations.description == "7") |
                      (raw_intensity.annotations.description == "3") |
                      (raw_intensity.annotations.description == "4") |
                      (raw_intensity.annotations.description == "6"))
raw_intensity.annotations.delete(unwanted)
raw_od = mne.preprocessing.nirs.optical_density(raw_intensity)
raw_od.plot(n_channels=len(raw_od.ch_names), duration=500, show_scrollbars=False)
raw_haemo = mne.preprocessing.nirs.beer_lambert_law(raw_od, ppf=0.1)

Reading 0 ... 5110  =      0.000 ...  1004.626 secs...


In [77]:
# Plot P>|z| 2nd Order' without interpolation 

fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
plot_glm_group_topo(raw_haemo.copy().pick(picks=range(12)),
                    df2,
                    value='p',
                    colorbar=True,
                    cmap='RdBu',
                    vlim=(0,.1),
                    axes=axes,
                   )
plot_glm_group_topo(raw_haemo.copy().pick(picks=range(12,22)),
                    df2,
                    value='p',
                    colorbar=True,
                    cmap='RdBu',
                    vlim=(0,.1),
                    axes=axes
                   )
plot_glm_group_topo(raw_haemo.copy().pick(picks=range(22,30)),
                    df2,
                    value='p',
                    colorbar=True,
                    cmap='RdBu',
                    vlim=(0,.1),
                    axes=axes
                   )
plot_glm_group_topo(raw_haemo.copy().pick(picks=range(30,38)),
                    df2,
                    value='p',
                    colorbar=True,
                    cmap='RdBu',
                    vlim=(0,.1),
                    axes=axes
                   )
axes.set_title('P>|z| 2nd Order p value')

KeyError: 'ch_name'

In [176]:
def sci(root, filename):
    # Construct the full path to the file
    subject_file_path = os.path.join(root, filename)
   
    # Check if the file exists
    if not os.path.exists(subject_file_path):
        print(f"File not found: {subject_file_path}")
        return None
   
    # Load the raw intensity data from the SNIRF file
    raw_intensity = mne.io.read_raw_snirf(subject_file_path, verbose=False)
    raw_od = mne.preprocessing.nirs.optical_density(raw_intensity)

    # Calculate the scalp coupling index
    sci_values = mne.preprocessing.nirs.scalp_coupling_index(raw_od)
   
    # Count channels with sci < 0.5
    low_sci_count = len(list(compress(raw_od.ch_names, sci_values < 0.5)))
   
    return filename, low_sci_count

# Define the root directory where the files are stored
root = r'C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\snirf'

# Get the list of all file names in the directory
subjects = [f for f in os.listdir(root) if f.endswith('.snirf')]

# Store results in a list
results = []

# Loop through each file in the directory and run the individual analysis on each
for file_name in subjects:
    result = sci(root, file_name)
    if result is not None:
        results.append(result)

# Print the results
for filename, low_sci_count in results:
    print(f"{filename}: {low_sci_count} channels with SCI < 0.5")

Reading 0 ... 5216  =      0.000 ...  1025.466 secs...
Reading 0 ... 4997  =      0.000 ...   982.410 secs...
Reading 0 ... 5276  =      0.000 ...  1037.262 secs...
Reading 0 ... 5165  =      0.000 ...  1015.439 secs...
Reading 0 ... 5110  =      0.000 ...  1004.626 secs...
Reading 0 ... 6180  =      0.000 ...  1214.988 secs...
Reading 0 ... 5225  =      0.000 ...  1027.235 secs...
Reading 0 ... 5438  =      0.000 ...  1069.111 secs...
Reading 0 ... 5208  =      0.000 ...  1023.893 secs...
Reading 0 ... 5175  =      0.000 ...  1017.405 secs...
Reading 0 ... 5330  =      0.000 ...  1047.878 secs...
Reading 0 ... 5450  =      0.000 ...  1071.470 secs...
Reading 0 ... 5385  =      0.000 ...  1058.691 secs...
Reading 0 ... 5736  =      0.000 ...  1127.698 secs...
Reading 0 ... 5194  =      0.000 ...  1021.140 secs...
Reading 0 ... 5244  =      0.000 ...  1030.970 secs...
Reading 0 ... 5318  =      0.000 ...  1045.519 secs...
Reading 0 ... 5331  =      0.000 ...  1048.075 secs...
Reading 0 